# Mission 3 ver7 — 새로운 KoELECTRA encoder 전체 미세조정

**기존 `mission3_ver5_inputs.zip`과 이 노트북 하나로 실행합니다.**

1. Drive 공유 문서함의 `mission3_ver5_inputs.zip`에서 **정리 → 바로가기 추가 → 내 드라이브**를 선택합니다.
2. 내 드라이브 바로 아래에 같은 이름의 바로가기를 둡니다.
3. 이 노트북을 Colab에서 열고 **GPU 런타임 → 모두 실행**을 선택합니다.
4. 바로가기를 만든 계정으로 Drive를 연결합니다. 다른 위치라면 아래 `INPUT_ZIP`만 바꾸세요.

ZIP의 기존 ver4 모델과 Training·Validation JSON을 그대로 복원합니다. WAV와 별도 DATA 폴더는 필요 없습니다.
새 encoder `monologg/koelectra-base-v3-discriminator`와 tokenizer는 공개 저장소에서 자동 다운로드합니다.
고정된 공식 커밋을 사용하며, 다운로드 단계 이후의 학습·추론은 로컬 파일만 읽습니다.
[모델 원본](https://huggingface.co/monologg/koelectra-base-v3-discriminator/tree/68b30cd259f34a4b5aa8786392612ba2a2617fcc).

**기존 입력:** `/content/mission3_ver5_inputs/DCC_AIdeas/results/mission3/ver4`  
**새 결과:** `내 드라이브/DCC_AIdeas/results/mission3/ver7`

ver7은 공개 사전학습 KoELECTRA에서 시작해 전체 encoder와 새 9-label attention 분류기를 함께 학습합니다.
기존 ver4의 5개 모델은 비교·혼합용으로 재사용합니다. 새 encoder 학습에는 frozen feature cache가 없습니다.
기본값은 최대 3 epoch, 개선 없는 2 epoch 후 중단입니다. 모델 선택은 Training 내부에서 끝냅니다.
최종 선택 결과와 새 후보 단독의 Validation 점수를 마지막에 각각 표시합니다.
후보 단독 점수는 진단용이며 최종 모델·혼합 비율·임계값을 바꾸지 않습니다.


In [ ]:
# 1. 기본 경로에서는 값을 바꾸지 않고 모두 실행하면 됩니다.
INPUT_ZIP = "/content/drive/MyDrive/mission3_ver5_inputs.zip"
BASE_DIR = None
TRAIN_LABEL_DIR = None
VALID_LABEL_DIR = None
OUTPUT_DIR = None           # Drive/MyDrive/DCC_AIdeas/results/mission3/ver7
INFERENCE_CACHE_DIR = None  # Drive의 공용 results/mission3/prediction_cache
PRETRAINED_DIR = None       # Colab 로컬의 고정 경로에 새 encoder 자동 준비
RUN_VALIDATION = True
RUN_CANDIDATE_DIAGNOSTIC = True  # 선택 확정 후 후보 단독 Validation 진단도 저장
TRAIN_CONFIG = {
    "bundle_base": False,
    "max_epochs": 3,
    "patience": 2,
    "learning_rate": 2e-5,
    "head_learning_rate": 1e-4,
    "stop_crossfit_splits": 3,
    "min_calibration_gain": 0.002,
    "min_calibration_improved_folds": 3,
}
INFERENCE_BATCH_SIZE = 8
# GPU 메모리에 맞춰 microbatch를 선택하고 유효 batch 32를 유지합니다.
# 설정을 바꾸어 새로 실험할 때는 OUTPUT_DIR을 새 폴더로 지정하세요.


In [ ]:
# 2. 기존 저장 모델과 새 KoELECTRA에 필요한 패키지 설치
import subprocess
import sys

REQUIREMENTS = ['torch>=2.6,<3', 'transformers==4.57.1', 'numpy==2.1.3', 'scikit-learn==1.6.1', 'scipy>=1.11,<2', 'iterative-stratification==0.1.9', 'joblib>=1.3,<2']
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q", *REQUIREMENTS], check=True)
print("환경 준비 완료. 모델 준비·학습·추론은 새 Python 프로세스에서 실행합니다.")


In [ ]:
# 3. Drive 연결 → 같은 입력 ZIP 검증·복원 → 경로 연결
import json
import os
import tempfile
import time
import uuid
from datetime import datetime, timedelta, timezone
from pathlib import Path

preparation_started = time.monotonic()
START_DIR = Path.cwd().resolve()
try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount("/content/drive", force_remount=False)
BOOTSTRAP_SOURCE = '"""Restore the one-file Mission 3 input bundle using only the standard library.\n\nThe uploaded ZIP remains in Drive. Extraction uses a fixed local directory so\nepoch resume sees the same initial checkpoint path across Colab sessions.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport stat\nimport tempfile\nimport zipfile\n\n\nBUNDLE_NAME = "mission3_ver5_inputs.zip"\nMODEL_PREFIX = "DCC_AIdeas/results/mission3/ver4/"\nTRAIN_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급/"\nVALID_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급/"\n\n\ndef _relative(name):\n    path = PurePosixPath(name)\n    if (not name or "\\\\" in name or ":" in name or path.is_absolute()\n            or ".." in path.parts or path.as_posix() != name.rstrip("/")):\n        raise ValueError(f"Unsafe bundle path: {name}")\n    return Path(*path.parts)\n\n\ndef _target(root, name):\n    path = root / _relative(name)\n    if not path.resolve().is_relative_to(root):\n        raise ValueError(f"Bundle path escapes extraction directory: {name}")\n    return path\n\n\ndef _hash(path):\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for data in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(data)\n    return digest.hexdigest()\n\n\ndef _files(manifest, archive):\n    if manifest.get("format") != "mission3_ver5_inputs_v1":\n        raise ValueError("Unsupported Mission 3 input bundle format")\n    files = manifest.get("files")\n    if not isinstance(files, list) or not files:\n        raise ValueError("Bundle manifest has no files")\n    names = set()\n    for row in files:\n        name = row["path"]\n        _relative(name)\n        if name in names or not name.startswith("DCC_AIdeas/"):\n            raise ValueError(f"Duplicate or unexpected bundle member: {name}")\n        names.add(name)\n        if (type(row["size"]) is not int or row["size"] < 0\n                or not re.fullmatch(r"[0-9a-f]{64}", row["sha256"])):\n            raise ValueError(f"Invalid manifest metadata: {name}")\n        info = archive.getinfo(name)\n        if info.is_dir() or info.file_size != row["size"]:\n            raise ValueError(f"Bundle size mismatch: {name}")\n    required = {MODEL_PREFIX + name for name in\n                ["mission3_ver4_model.ckpt", "splits.json", "calibration_predictions.npz", "sparse.joblib",\n                 *[f"fold_{i}.ckpt" for i in range(1, 6)], "tokenizer/tokenizer.json",\n                 "tokenizer/tokenizer_config.json", "tokenizer/vocab.txt", "tokenizer/special_tokens_map.json"]}\n    if not required <= names:\n        raise ValueError(f"Missing ver4 inputs: {sorted(required - names)}")\n    for prefix, key in [(TRAIN_PREFIX, "training_count"), (VALID_PREFIX, "validation_count")]:\n        count = sum(name.startswith(prefix) and name.endswith(".json") for name in names)\n        if count < 1 or type(manifest.get(key)) is not int or count != manifest[key]:\n            raise ValueError(f"Bundle JSON count mismatch: {key}")\n    infos = archive.infolist()\n    if len({info.filename for info in infos}) != len(infos):\n        raise ValueError("Duplicate ZIP member")\n    allowed_files = names | {"bundle_manifest.json", "README_UPLOAD.txt"}\n    for info in infos:\n        _relative(info.filename)\n        if stat.S_ISLNK(info.external_attr >> 16) or info.flag_bits & 1:\n            raise ValueError("Links and encrypted ZIP entries are not supported")\n        if not info.is_dir() and info.filename not in allowed_files:\n            raise ValueError(f"Unlisted bundle member: {info.filename}")\n    return files\n\n\ndef restore_input_bundle(archive_path, destination):\n    """Verify SHA256 while extracting; reuse unchanged files from a verified run."""\n    archive_path = Path(archive_path).expanduser().resolve()\n    root = Path(destination).expanduser().resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    marker = root / ".mission3_bundle_complete.json"\n    with zipfile.ZipFile(archive_path) as archive:\n        manifest_bytes = archive.read("bundle_manifest.json")\n        manifest = json.loads(manifest_bytes)\n        files = _files(manifest, archive)\n        signature = hashlib.sha256(manifest_bytes).hexdigest()\n        previous = {}\n        try:\n            saved = json.loads(marker.read_text(encoding="utf-8"))\n            if saved.get("manifest_sha256") == signature:\n                previous = saved["files"]\n        except (OSError, ValueError, KeyError):\n            pass\n        pending, state = [], {}\n        for row in files:\n            target = _target(root, row["path"])\n            current = target.stat() if target.is_file() else None\n            stamp = [current.st_size, current.st_mtime_ns] if current else None\n            if stamp and current.st_size == row["size"] and previous.get(row["path"]) == stamp:\n                state[row["path"]] = stamp\n            else:\n                pending.append(row)\n        if not pending:\n            print("입력 ZIP: 이전에 검증한 Colab 로컬 파일을 재사용합니다.", flush=True)\n            return root / "DCC_AIdeas"\n        if shutil.disk_usage(root).free < sum(row["size"] for row in pending):\n            raise OSError("Colab 로컬 저장소에 입력 ZIP을 풀 공간이 부족합니다.")\n        print(f"입력 ZIP 확인·압축 해제: {len(pending):,}개 파일 (WAV 없음)", flush=True)\n        for index, row in enumerate(pending, 1):\n            target = _target(root, row["path"])\n            target.parent.mkdir(parents=True, exist_ok=True)\n            # Recover completed files from an interrupted extraction without copying them again.\n            verified = target.is_file() and target.stat().st_size == row["size"] and _hash(target) == row["sha256"]\n            if not verified:\n                temporary = None\n                try:\n                    digest, size = hashlib.sha256(), 0\n                    with archive.open(row["path"]) as reader, tempfile.NamedTemporaryFile(\n                            dir=target.parent, prefix=".extract_", suffix=".tmp", delete=False) as writer:\n                        temporary = Path(writer.name)\n                        for chunk in iter(lambda: reader.read(8 * 1024 * 1024), b""):\n                            writer.write(chunk)\n                            digest.update(chunk)\n                            size += len(chunk)\n                    if size != row["size"] or digest.hexdigest() != row["sha256"]:\n                        raise ValueError(f"ZIP 내용 검증 실패: {row[\'path\']}")\n                    temporary.replace(target)\n                finally:\n                    if temporary is not None:\n                        temporary.unlink(missing_ok=True)\n            current = target.stat()\n            state[row["path"]] = [current.st_size, current.st_mtime_ns]\n            if row["size"] > 100 * 1024 * 1024 or index % 2000 == 0 or index == len(pending):\n                print(f"압축 해제 {index:,}/{len(pending):,}: {target.name}", flush=True)\n        # A marker is published only after every file has passed hash verification.\n        temporary = marker.with_suffix(".tmp")\n        temporary.write_text(json.dumps({"manifest_sha256": signature, "files": state}), encoding="utf-8")\n        temporary.replace(marker)\n    print(f"준비 완료: Training {manifest[\'training_count\']:,}개 / Validation {manifest[\'validation_count\']:,}개", flush=True)\n    return root / "DCC_AIdeas"\n'
bootstrap_namespace = {}
exec(compile(BOOTSTRAP_SOURCE, "mission3_bootstrap.py", "exec"), bootstrap_namespace)
bundle_path = Path(INPUT_ZIP).expanduser().resolve() if INPUT_ZIP is not None else None
BUNDLE_ROOT = None
if bundle_path is not None:
    if not bundle_path.is_file():
        raise FileNotFoundError(
            f"ZIP 바로가기를 찾지 못했습니다: {bundle_path}\n"
            "공유 ZIP의 '정리 → 바로가기 추가 → 내 드라이브'를 선택하고 INPUT_ZIP을 확인하세요. "
            "바로가기를 만든 계정으로 연결하세요. 연결 후 바로가기를 추가했다면 "
            "drive.mount의 force_remount=True로 한 번 재연결하세요.")
    extract_dir = (Path("/content/mission3_ver5_inputs") if IN_COLAB
                   else Path(tempfile.gettempdir()) / "mission3_ver5_inputs")
    BUNDLE_ROOT = bootstrap_namespace["restore_input_bundle"](bundle_path, extract_dir)
    BASE_DIR = BUNDLE_ROOT / "results/mission3/ver4"
if BASE_DIR is None:
    raise ValueError("INPUT_ZIP 또는 기존 ver4 폴더인 BASE_DIR를 지정하세요.")
BASE_DIR = Path(BASE_DIR).expanduser().resolve()
for name in ("mission3_ver4_model.ckpt", "splits.json", "calibration_predictions.npz",
             "sparse.joblib", "tokenizer", *[f"fold_{i}.ckpt" for i in range(1, 6)]):
    if not (BASE_DIR / name).exists():
        raise FileNotFoundError(f"기존 ver4 입력 파일이 빠졌습니다: {BASE_DIR / name}")
data_root = ((BUNDLE_ROOT if BUNDLE_ROOT is not None else BASE_DIR.parents[2])
             / "DATA/대학부 데이터")
def labels(explicit, split, name, required=True):
    path = (data_root / split / "2.라벨링데이터" / name if explicit is None
            else Path(explicit).expanduser()).resolve()
    if path.is_dir() and next(path.rglob("*.json"), None) is not None:
        return path
    if required:
        raise FileNotFoundError(f"{split} JSON 경로를 확인하세요: {path}")
    return None
TRAIN_LABEL_DIR = labels(TRAIN_LABEL_DIR, "Training", "TL_서울_구급")
VALID_LABEL_DIR = labels(VALID_LABEL_DIR, "Validation", "VL_서울_구급", RUN_VALIDATION)
result_parent = (Path("/content/drive/MyDrive/DCC_AIdeas/results/mission3") if IN_COLAB
                 else START_DIR / "results/mission3")
OUTPUT_DIR = (result_parent / "ver7" if OUTPUT_DIR is None else Path(OUTPUT_DIR).expanduser()).resolve()
INFERENCE_CACHE_DIR = (result_parent / "prediction_cache" if INFERENCE_CACHE_DIR is None
                       else Path(INFERENCE_CACHE_DIR).expanduser()).resolve()
local_pretrained = ((Path("/content") if IN_COLAB else Path(tempfile.gettempdir()))
                    / "mission3_ver7_pretrained/koelectra_base_v3")
PRETRAINED_DIR = (local_pretrained if PRETRAINED_DIR is None else Path(PRETRAINED_DIR).expanduser()).resolve()
if OUTPUT_DIR == BASE_DIR or OUTPUT_DIR.is_relative_to(BASE_DIR) or BASE_DIR.is_relative_to(OUTPUT_DIR):
    raise ValueError("OUTPUT_DIR는 기존 ver4 입력 폴더와 분리된 새 폴더여야 합니다.")
PREPARATION_MINUTES = (time.monotonic() - preparation_started) / 60
print("입력 ZIP:", bundle_path)
print("기존 ver4 모델:", BASE_DIR)
print("Training JSON:", TRAIN_LABEL_DIR)
print("최종 평가 JSON:", VALID_LABEL_DIR)
print("새 KoELECTRA 로컬 경로:", PRETRAINED_DIR)
print("새 ver7 결과 (Drive):", OUTPUT_DIR)
print("공유 원시 예측 캐시 (Drive):", INFERENCE_CACHE_DIR)
print(f"Drive 연결·입력 준비: {PREPARATION_MINUTES:.1f}분")


In [ ]:
# @title 4. 노트북에 포함된 실행 코드 준비 (별도 소스 업로드 불필요)
RUNTIME_SOURCES = {'__init__.py': '"""Mission 3 ver7: fresh public KoELECTRA finetuning and calibrated ensemble."""\n', 'bootstrap.py': '"""Restore the one-file Mission 3 input bundle using only the standard library.\n\nThe uploaded ZIP remains in Drive. Extraction uses a fixed local directory so\nepoch resume sees the same initial checkpoint path across Colab sessions.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport stat\nimport tempfile\nimport zipfile\n\n\nBUNDLE_NAME = "mission3_ver5_inputs.zip"\nMODEL_PREFIX = "DCC_AIdeas/results/mission3/ver4/"\nTRAIN_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급/"\nVALID_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급/"\n\n\ndef _relative(name):\n    path = PurePosixPath(name)\n    if (not name or "\\\\" in name or ":" in name or path.is_absolute()\n            or ".." in path.parts or path.as_posix() != name.rstrip("/")):\n        raise ValueError(f"Unsafe bundle path: {name}")\n    return Path(*path.parts)\n\n\ndef _target(root, name):\n    path = root / _relative(name)\n    if not path.resolve().is_relative_to(root):\n        raise ValueError(f"Bundle path escapes extraction directory: {name}")\n    return path\n\n\ndef _hash(path):\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for data in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(data)\n    return digest.hexdigest()\n\n\ndef _files(manifest, archive):\n    if manifest.get("format") != "mission3_ver5_inputs_v1":\n        raise ValueError("Unsupported Mission 3 input bundle format")\n    files = manifest.get("files")\n    if not isinstance(files, list) or not files:\n        raise ValueError("Bundle manifest has no files")\n    names = set()\n    for row in files:\n        name = row["path"]\n        _relative(name)\n        if name in names or not name.startswith("DCC_AIdeas/"):\n            raise ValueError(f"Duplicate or unexpected bundle member: {name}")\n        names.add(name)\n        if (type(row["size"]) is not int or row["size"] < 0\n                or not re.fullmatch(r"[0-9a-f]{64}", row["sha256"])):\n            raise ValueError(f"Invalid manifest metadata: {name}")\n        info = archive.getinfo(name)\n        if info.is_dir() or info.file_size != row["size"]:\n            raise ValueError(f"Bundle size mismatch: {name}")\n    required = {MODEL_PREFIX + name for name in\n                ["mission3_ver4_model.ckpt", "splits.json", "calibration_predictions.npz", "sparse.joblib",\n                 *[f"fold_{i}.ckpt" for i in range(1, 6)], "tokenizer/tokenizer.json",\n                 "tokenizer/tokenizer_config.json", "tokenizer/vocab.txt", "tokenizer/special_tokens_map.json"]}\n    if not required <= names:\n        raise ValueError(f"Missing ver4 inputs: {sorted(required - names)}")\n    for prefix, key in [(TRAIN_PREFIX, "training_count"), (VALID_PREFIX, "validation_count")]:\n        count = sum(name.startswith(prefix) and name.endswith(".json") for name in names)\n        if count < 1 or type(manifest.get(key)) is not int or count != manifest[key]:\n            raise ValueError(f"Bundle JSON count mismatch: {key}")\n    infos = archive.infolist()\n    if len({info.filename for info in infos}) != len(infos):\n        raise ValueError("Duplicate ZIP member")\n    allowed_files = names | {"bundle_manifest.json", "README_UPLOAD.txt"}\n    for info in infos:\n        _relative(info.filename)\n        if stat.S_ISLNK(info.external_attr >> 16) or info.flag_bits & 1:\n            raise ValueError("Links and encrypted ZIP entries are not supported")\n        if not info.is_dir() and info.filename not in allowed_files:\n            raise ValueError(f"Unlisted bundle member: {info.filename}")\n    return files\n\n\ndef restore_input_bundle(archive_path, destination):\n    """Verify SHA256 while extracting; reuse unchanged files from a verified run."""\n    archive_path = Path(archive_path).expanduser().resolve()\n    root = Path(destination).expanduser().resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    marker = root / ".mission3_bundle_complete.json"\n    with zipfile.ZipFile(archive_path) as archive:\n        manifest_bytes = archive.read("bundle_manifest.json")\n        manifest = json.loads(manifest_bytes)\n        files = _files(manifest, archive)\n        signature = hashlib.sha256(manifest_bytes).hexdigest()\n        previous = {}\n        try:\n            saved = json.loads(marker.read_text(encoding="utf-8"))\n            if saved.get("manifest_sha256") == signature:\n                previous = saved["files"]\n        except (OSError, ValueError, KeyError):\n            pass\n        pending, state = [], {}\n        for row in files:\n            target = _target(root, row["path"])\n            current = target.stat() if target.is_file() else None\n            stamp = [current.st_size, current.st_mtime_ns] if current else None\n            if stamp and current.st_size == row["size"] and previous.get(row["path"]) == stamp:\n                state[row["path"]] = stamp\n            else:\n                pending.append(row)\n        if not pending:\n            print("입력 ZIP: 이전에 검증한 Colab 로컬 파일을 재사용합니다.", flush=True)\n            return root / "DCC_AIdeas"\n        if shutil.disk_usage(root).free < sum(row["size"] for row in pending):\n            raise OSError("Colab 로컬 저장소에 입력 ZIP을 풀 공간이 부족합니다.")\n        print(f"입력 ZIP 확인·압축 해제: {len(pending):,}개 파일 (WAV 없음)", flush=True)\n        for index, row in enumerate(pending, 1):\n            target = _target(root, row["path"])\n            target.parent.mkdir(parents=True, exist_ok=True)\n            # Recover completed files from an interrupted extraction without copying them again.\n            verified = target.is_file() and target.stat().st_size == row["size"] and _hash(target) == row["sha256"]\n            if not verified:\n                temporary = None\n                try:\n                    digest, size = hashlib.sha256(), 0\n                    with archive.open(row["path"]) as reader, tempfile.NamedTemporaryFile(\n                            dir=target.parent, prefix=".extract_", suffix=".tmp", delete=False) as writer:\n                        temporary = Path(writer.name)\n                        for chunk in iter(lambda: reader.read(8 * 1024 * 1024), b""):\n                            writer.write(chunk)\n                            digest.update(chunk)\n                            size += len(chunk)\n                    if size != row["size"] or digest.hexdigest() != row["sha256"]:\n                        raise ValueError(f"ZIP 내용 검증 실패: {row[\'path\']}")\n                    temporary.replace(target)\n                finally:\n                    if temporary is not None:\n                        temporary.unlink(missing_ok=True)\n            current = target.stat()\n            state[row["path"]] = [current.st_size, current.st_mtime_ns]\n            if row["size"] > 100 * 1024 * 1024 or index % 2000 == 0 or index == len(pending):\n                print(f"압축 해제 {index:,}/{len(pending):,}: {target.name}", flush=True)\n        # A marker is published only after every file has passed hash verification.\n        temporary = marker.with_suffix(".tmp")\n        temporary.write_text(json.dumps({"manifest_sha256": signature, "files": state}), encoding="utf-8")\n        temporary.replace(marker)\n    print(f"준비 완료: Training {manifest[\'training_count\']:,}개 / Validation {manifest[\'validation_count\']:,}개", flush=True)\n    return root / "DCC_AIdeas"\n', 'neural.py': '"""One bounded full-encoder KoELECTRA run with direct token label attention.\n\nEvery window is encoded live from epoch one. The only loss is fit-weighted\nBCE; Training-stop grouped threshold cross-fitting chooses the checkpoint.\nEpoch zero is explicitly the public encoder with a newly initialized head,\nnot evidence that a fine-tuning epoch was completed.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport tempfile\nimport time\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn.utils.rnn import pad_sequence\nfrom transformers import AutoConfig, AutoModel, get_linear_schedule_with_warmup\n\nif __package__:\n    from .selection import StopSelector, better\n    from .ver4_runtime.neural import NUM_LABELS, DialogueCollator, _device, _loader, _optimizer, _targets, tokenize_dialogues\nelse:\n    from selection import StopSelector, better\n    from ver4_runtime.neural import NUM_LABELS, DialogueCollator, _device, _loader, _optimizer, _targets, tokenize_dialogues\n\n\nMODEL_FORMAT = "mission3_ver7_neural"\nLAST_FORMAT = "mission3_ver7_last"\nDEFAULTS = {\n    "model_name": "monologg/koelectra-base-v3-discriminator",\n    "max_length": 512, "stride": 64, "max_epochs": 3, "patience": 2,\n    "stop_crossfit_splits": 3, "learning_rate": 2e-5, "head_learning_rate": 1e-4,\n    "attention_dim": 256, "dropout": 0.1, "batch_size": 4, "window_batch_size": 4,\n    "gradient_accumulation_steps": 8, "inference_batch_size": 8,\n    "gradient_checkpointing": True, "device": "cuda", "log_every": 100,\n    "layerwise_lr_decay": 0.95, "weight_decay": 0.01, "warmup_ratio": 0.1,\n}\n\n\ndef native_bf16_supported():\n    if not torch.cuda.is_available():\n        return False\n    try:\n        return bool(torch.cuda.is_bf16_supported(including_emulation=False))\n    except TypeError:\n        return torch.cuda.get_device_capability()[0] >= 8\n\n\ndef _amp(device):\n    return torch.amp.autocast(device_type=device.type,\n                              dtype=torch.bfloat16 if device.type == "cuda" and native_bf16_supported() else torch.float16,\n                              enabled=device.type == "cuda")\n\n\ndef execution_identity(device):\n    """Exact epoch resume requires the same numerical execution environment."""\n    device = torch.device(device)\n    identity = {"device_type": device.type, "torch": str(torch.__version__),\n                "transformers": importlib.metadata.version("transformers"),\n                "cuda_version": torch.version.cuda,\n                "amp_dtype": "float32" if device.type != "cuda" else "bfloat16" if native_bf16_supported() else "float16",\n                "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),\n                "matmul_allow_tf32": torch.backends.cuda.matmul.allow_tf32,\n                "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32}\n    if device.type == "cuda":\n        identity.update(gpu=torch.cuda.get_device_name(device),\n                        capability=list(torch.cuda.get_device_capability(device)),\n                        cuda_device_count=torch.cuda.device_count())\n    else:\n        identity["cpu_threads"] = torch.get_num_threads()\n    return identity\n\n\ndef _settings(config):\n    settings = {name: config.get(name, value) for name, value in DEFAULTS.items()}\n    for name in ("max_epochs", "patience", "stop_crossfit_splits", "attention_dim", "batch_size",\n                 "window_batch_size", "gradient_accumulation_steps", "inference_batch_size", "max_length", "stride", "log_every"):\n        minimum = 0 if name in ("max_epochs", "stride", "log_every") else 2 if name == "stop_crossfit_splits" else 1\n        if isinstance(settings[name], bool) or not isinstance(settings[name], int) or settings[name] < minimum:\n            raise ValueError(f"{name} must be an integer >= {minimum}")\n    if settings["max_epochs"] > 3:\n        raise ValueError("ver7 is bounded to max_epochs <= 3")\n    if settings["max_length"] > 512 or settings["max_length"] < 3 or settings["stride"] >= settings["max_length"] - 2:\n        raise ValueError("max_length must be in [3, 512]; stride must fit the semantic token window")\n    for name in ("learning_rate", "head_learning_rate", "layerwise_lr_decay"):\n        if isinstance(settings[name], bool) or not math.isfinite(float(settings[name])) or float(settings[name]) <= 0:\n            raise ValueError(f"{name} must be positive and finite")\n    if not 0 < settings["layerwise_lr_decay"] <= 1:\n        raise ValueError("layerwise_lr_decay must be in (0, 1]")\n    if not 0 <= float(settings["dropout"]) < 1 or not 0 <= float(settings["warmup_ratio"]) < 1:\n        raise ValueError("dropout/warmup_ratio must lie in [0, 1)")\n    if not math.isfinite(float(settings["weight_decay"])) or settings["weight_decay"] < 0:\n        raise ValueError("weight_decay must be nonnegative and finite")\n    if not isinstance(settings["gradient_checkpointing"], bool):\n        raise ValueError("gradient_checkpointing must be a boolean")\n    if not isinstance(settings["model_name"], str) or not settings["model_name"]:\n        raise ValueError("model_name must be nonempty")\n    return settings\n\n\nclass LabelAttentionHead(nn.Module):\n    """Each label directly attends to semantic tokens across all windows."""\n    def __init__(self, hidden_size, num_labels=NUM_LABELS, dropout=0.1, attention_dim=256):\n        super().__init__()\n        self.token_projection = nn.Linear(hidden_size, attention_dim)\n        self.token_queries = nn.Parameter(torch.empty(num_labels, attention_dim))\n        self.label_weights = nn.Parameter(torch.empty(num_labels, hidden_size))\n        self.label_bias = nn.Parameter(torch.zeros(num_labels))\n        self.dropout = nn.Dropout(dropout)\n        nn.init.normal_(self.token_queries, std=0.02)\n        nn.init.normal_(self.label_weights, std=0.02)\n\n    def forward(self, documents):\n        rows, lengths = [], []\n        for windows in documents:\n            tokens = torch.cat([window["hidden"][window["content_mask"]] for window in windows], dim=0)\n            lengths.append(len(tokens))\n            # Dummy zero state gives empty calls a finite bias-only prediction.\n            rows.append(tokens if len(tokens) else windows[0]["hidden"].new_zeros(1, windows[0]["hidden"].shape[-1]))\n        hidden = pad_sequence(rows, batch_first=True)\n        lengths = torch.tensor(lengths, device=hidden.device)\n        mask = torch.arange(hidden.shape[1], device=hidden.device)[None, :] < lengths[:, None]\n        projected = torch.tanh(self.token_projection(self.dropout(hidden)))\n        scores = (projected @ self.token_queries.T).float()\n        attention = scores.masked_fill(~mask[:, :, None], torch.finfo(scores.dtype).min).softmax(1)\n        attention = attention * mask[:, :, None]\n        attention = attention / attention.sum(1, keepdim=True).clamp_min(1e-12)\n        pooled = torch.einsum("btl,bth->blh", attention.to(hidden.dtype), hidden)\n        return (self.dropout(pooled) * self.label_weights).sum(-1) + self.label_bias\n\n\nclass DialogueClassifier(nn.Module):\n    def __init__(self, encoder, num_labels=NUM_LABELS, dropout=0.1, window_batch_size=4,\n                 attention_dim=256, special_token_ids=()):\n        super().__init__()\n        if int(window_batch_size) < 1:\n            raise ValueError("window_batch_size must be positive")\n        self.encoder, self.window_batch_size = encoder, int(window_batch_size)\n        self.special_token_ids = tuple(int(x) for x in special_token_ids)\n        self.head = LabelAttentionHead(encoder.config.hidden_size, num_labels, dropout, attention_dim)\n\n    def forward(self, doc_indices, **inputs):\n        documents = [[] for _ in range(int(doc_indices.max().item()) + 1)]\n        for start in range(0, len(doc_indices), self.window_batch_size):\n            subset = {key: value[start:start + self.window_batch_size] for key, value in inputs.items()}\n            hidden = self.encoder(**subset).last_hidden_state\n            for offset in range(len(hidden)):\n                valid = subset["attention_mask"][offset].bool()\n                states = hidden[offset][valid]\n                ids = subset["input_ids"][offset][valid]\n                content = torch.ones(len(ids), dtype=torch.bool, device=ids.device)\n                for token_id in self.special_token_ids:\n                    content &= ids.ne(token_id)\n                documents[int(doc_indices[start + offset])].append({"hidden": states, "content_mask": content})\n        return self.head(documents)\n\n\ndef _configure_training(model, settings):\n    model.requires_grad_(True)\n    if not all(parameter.requires_grad for parameter in model.encoder.parameters()):\n        raise RuntimeError("The complete ver7 encoder must train from epoch one")\n    if settings["gradient_checkpointing"]:\n        model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n    if hasattr(model.encoder.config, "use_cache"):\n        model.encoder.config.use_cache = False\n\n\ndef _positive_weights(targets):\n    positives = np.asarray(targets, dtype=np.float32).sum(axis=0)\n    ratio = (len(targets) - positives) / positives.clip(min=1)\n    return np.clip(np.sqrt(ratio.clip(min=1)), 1, 5).astype(np.float32)\n\n\ndef _read_checkpoint(path):\n    return torch.load(Path(path), map_location="cpu", weights_only=True, mmap=True)\n\n\ndef _encoder(configuration):\n    values = dict(configuration)\n    return AutoModel.from_config(AutoConfig.for_model(values.pop("model_type"), **values))\n\n\ndef _special_ids(tokenizer):\n    return sorted({int(value) for name in ("pad_token_id", "cls_token_id", "sep_token_id", "bos_token_id", "eos_token_id")\n                   if (value := getattr(tokenizer, name, None)) is not None})\n\n\ndef _atomic_save(value, path):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    descriptor, temporary = tempfile.mkstemp(prefix=path.name + ".", suffix=".tmp", dir=path.parent)\n    os.close(descriptor)\n    try:\n        torch.save(value, temporary)\n        os.replace(temporary, path)\n    finally:\n        if os.path.exists(temporary):\n            os.unlink(temporary)\n\n\ndef _rng_state(generator):\n    numpy_state = np.random.get_state()\n    return {"python": random.getstate(),\n            "numpy": {"name": numpy_state[0], "keys": numpy_state[1].tolist(), "position": int(numpy_state[2]),\n                      "has_gauss": int(numpy_state[3]), "cached_gaussian": float(numpy_state[4])},\n            "torch": torch.get_rng_state(), "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],\n            "loader": generator.get_state()}\n\n\ndef _restore_rng(state, generator):\n    random.setstate(state["python"])\n    values = state["numpy"]\n    np.random.set_state((values["name"], np.asarray(values["keys"], dtype=np.uint32), values["position"],\n                         values["has_gauss"], values["cached_gaussian"]))\n    torch.set_rng_state(state["torch"])\n    if state["cuda"]:\n        if not torch.cuda.is_available() or len(state["cuda"]) != torch.cuda.device_count():\n            raise ValueError("CUDA device count changed; exact epoch resume is unavailable")\n        torch.cuda.set_rng_state_all(state["cuda"])\n    generator.set_state(state["loader"])\n\n\ndef _seed(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef _cpu_state(module):\n    return {key: value.detach().cpu().clone() for key, value in module.state_dict().items()}\n\n\ndef _check_identity(state, signature, fingerprint, execution):\n    if state.get("run_signature") != signature:\n        raise ValueError("Checkpoint run signature mismatch; use a new output directory")\n    if state.get("execution") != execution:\n        raise ValueError("Exact epoch resume requires the same execution environment, GPU and precision; use a new output directory")\n    if state.get("configuration_fingerprint") != fingerprint:\n        raise ValueError("Checkpoint config fingerprint mismatch; use a new output directory")\n\n\n@torch.inference_mode()\ndef _predict(model, loader, device):\n    model.eval()\n    probabilities = []\n    for batch in loader:\n        batch.pop("labels", None)\n        batch = {key: value.to(device) for key, value in batch.items()}\n        with _amp(device):\n            logits = model(**batch)\n        if not torch.isfinite(logits).all():\n            raise FloatingPointError("Prediction logits contain nonfinite values")\n        probabilities.append(logits.float().sigmoid().cpu().numpy())\n    return np.concatenate(probabilities) if probabilities else np.empty((0, NUM_LABELS), dtype=np.float32)\n\n\ndef _from_saved(checkpoint, config):\n    if checkpoint.get("format") not in (MODEL_FORMAT, LAST_FORMAT) or checkpoint.get("format_version") != 1:\n        raise ValueError("Unsupported ver7 neural checkpoint")\n    architecture = dict(checkpoint["architecture"])\n    architecture["window_batch_size"] = int(config.get("window_batch_size", architecture["window_batch_size"]))\n    model = DialogueClassifier(_encoder(checkpoint["encoder_config"]), **architecture)\n    model.load_state_dict(checkpoint["model_state"], strict=True)\n    return model\n\n\ndef predict_checkpoint(checkpoint_path, documents, tokenizer, config):\n    """Reconstruct solely from saved config and weights, fully offline."""\n    checkpoint = _read_checkpoint(checkpoint_path)\n    if checkpoint.get("format") != MODEL_FORMAT:\n        raise ValueError("Inference requires the selected best ver7 checkpoint")\n    device = _device(config)\n    model = _from_saved(checkpoint, config).to(device)\n    del checkpoint\n    probabilities = _predict(model, _loader(documents, None, tokenizer, config.get("inference_batch_size", 8)), device)\n    if not np.isfinite(probabilities).all():\n        raise FloatingPointError("Inference produced nonfinite probabilities")\n    return probabilities.astype(np.float32, copy=False)\n\n\ndef _best_payload(last):\n    return {**last, "format": MODEL_FORMAT, "epoch": last["best_epoch"],\n            "stop_ap": last["best_score"]["stop_macro_ap"],\n            "stop_macro_f1": last["best_score"]["stop_macro_f1"]}\n\n\ndef _result(state, resumed, last_path):\n    return {"best_epoch": state["best_epoch"], "best_stop_ap": state["best_score"]["stop_macro_ap"],\n            "best_stop_macro_f1": state["best_score"]["stop_macro_f1"], "history": state["history"],\n            "completed_epochs": state["completed_epochs"], "resumed": resumed, "last_checkpoint": str(last_path),\n            "execution": state["execution"], "best_is_initial_baseline": state["best_epoch"] == 0,\n            "selection_scope": "Original Training fold stop: grouped threshold cross-fit macro-F1; AP secondary. "\n                               "This is reused epoch selection, not model-level OOF or official Validation."}\n\n\ndef train_model(train_documents, train_targets, stop_documents, stop_targets, tokenizer, config,\n                checkpoint_path, seed, initial_model_dir, run_signature, stop_groups=None):\n    """Train once from local public weights, resume only committed epoch state.\n\nThe caller\'s run_signature binds source data, tokenizer and pretrained-file\nhashes. This module additionally binds token windows, targets, groups, settings,\nseed and execution identity. max_epochs=0 records only the initial baseline.\n    """\n    train_targets = _targets(train_targets, train_documents, "fit")\n    stop_targets = _targets(stop_targets, stop_documents, "stop")\n    if stop_groups is None or np.asarray(stop_groups).shape != (len(stop_documents),):\n        raise ValueError("stop_groups must match stop documents")\n    if not isinstance(run_signature, str) or not run_signature:\n        raise ValueError("run_signature must be a nonempty string")\n    settings = _settings(config)\n    device = _device(settings)\n    execution = execution_identity(device)\n    fingerprint_data = {"settings": settings, "seed": int(seed),\n                        "stop_groups": list(map(str, stop_groups)), "execution": execution}\n    identity_hash = hashlib.sha256(json.dumps(fingerprint_data, sort_keys=True).encode())\n    for documents, targets in ((train_documents, train_targets), (stop_documents, stop_targets)):\n        for document in documents:\n            identity_hash.update(json.dumps(document, sort_keys=True, separators=(",", ":")).encode())\n            identity_hash.update(b"\\n")\n        identity_hash.update(targets.tobytes())\n    fingerprint = identity_hash.hexdigest()\n    checkpoint_path = Path(checkpoint_path)\n    last_path = checkpoint_path.with_name(checkpoint_path.stem + "_last.ckpt")\n    last = _read_checkpoint(last_path) if last_path.exists() else None\n    best_saved = _read_checkpoint(checkpoint_path) if checkpoint_path.exists() else None\n    for saved in (last, best_saved):\n        if saved is not None:\n            _check_identity(saved, run_signature, fingerprint, execution)\n    if best_saved is not None and best_saved.get("format") != MODEL_FORMAT:\n        raise ValueError("Unsupported ver7 best checkpoint")\n    if last is not None:\n        if last.get("format") != LAST_FORMAT:\n            raise ValueError("Unsupported ver7 resume checkpoint")\n        best_matches = (best_saved is not None and best_saved.get("epoch") == last["best_epoch"] and\n                        best_saved.get("stop_macro_f1") == last["best_score"]["stop_macro_f1"] and\n                        best_saved.get("stop_ap") == last["best_score"]["stop_macro_ap"])\n        if not best_matches:\n            if last["best_epoch"] != last["completed_epochs"]:\n                raise ValueError("Best checkpoint is missing/inconsistent with completed epoch state")\n            _atomic_save(_best_payload(last), checkpoint_path)\n        if last["completed"]:\n            return _result(last, True, last_path)\n    elif best_saved is not None:\n        raise ValueError("Best checkpoint exists without resume state")\n    del best_saved\n    selector = StopSelector(stop_targets, stop_groups, seed=seed, n_splits=settings["stop_crossfit_splits"])\n    resumed = last is not None\n    _seed(seed)\n    if last is None:\n        directory = Path(initial_model_dir)\n        if not directory.is_dir():\n            raise ValueError("initial_model_dir must be an already downloaded local public model directory")\n        encoder = AutoModel.from_pretrained(str(directory), local_files_only=True)\n        architecture = {"num_labels": NUM_LABELS, "dropout": settings["dropout"],\n                        "window_batch_size": settings["window_batch_size"], "attention_dim": settings["attention_dim"],\n                        "special_token_ids": _special_ids(tokenizer)}\n        model = DialogueClassifier(encoder, **architecture)\n        metadata = {"encoder_config": encoder.config.to_dict(), "architecture": architecture,\n                    "model_name": settings["model_name"], "initialization": "local_public_pretrained_encoder_and_new_label_attention_head"}\n    else:\n        model = _from_saved(last, settings)\n        metadata = {key: last[key] for key in ("encoder_config", "architecture", "model_name", "initialization")}\n    model.to(device)\n    _configure_training(model, settings)\n    loader = _loader(train_documents, train_targets, tokenizer, settings["batch_size"], shuffle=True, seed=seed)\n    stop_loader = _loader(stop_documents, None, tokenizer, settings["inference_batch_size"])\n    pos_weight = _positive_weights(train_targets)\n    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.as_tensor(pos_weight, device=device))\n    optimizer = _optimizer(model, settings)\n    accumulation = settings["gradient_accumulation_steps"]\n    total_steps = math.ceil(len(loader) / accumulation) * max(1, settings["max_epochs"])\n    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * settings["warmup_ratio"]), total_steps)\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and not native_bf16_supported())\n    if last is None:\n        started = time.monotonic()\n        best_score = selector.score(_predict(model, stop_loader, device))\n        history = [{"epoch": 0, "phase": "initial_public_encoder", "train_loss": None, **best_score,\n                    "minutes": (time.monotonic() - started) / 60}]\n        completed, best_epoch, stale = 0, 0, 0\n    else:\n        optimizer.load_state_dict(last["optimizer"])\n        scheduler.load_state_dict(last["scheduler"])\n        scaler.load_state_dict(last["scaler"])\n        _restore_rng(last["rng"], loader.generator)\n        best_score, best_epoch, stale = last["best_score"], last["best_epoch"], last["stale_epochs"]\n        history, completed = list(last["history"]), last["completed_epochs"]\n    del last\n\n    def commit(epoch, done, improved):\n        state = {"format": LAST_FORMAT, "format_version": 1, **metadata, "model_state": _cpu_state(model),\n                 "execution": execution, "seed": int(seed), "run_signature": run_signature,\n                 "configuration_fingerprint": fingerprint, "settings": settings,\n                 "completed_epochs": epoch, "completed": done, "best_epoch": best_epoch,\n                 "best_score": best_score, "stale_epochs": stale, "history": history,\n                 "fit_positive_weights": pos_weight.tolist(), "optimizer": optimizer.state_dict(),\n                 "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(), "rng": _rng_state(loader.generator)}\n        # Last commits first. If the best write is interrupted, it can be\n        # rebuilt from last iff this epoch was the selected improvement.\n        _atomic_save(state, last_path)\n        if improved:\n            _atomic_save(_best_payload(state), checkpoint_path)\n        return state\n\n    if not resumed:\n        state = commit(0, settings["max_epochs"] == 0, True)\n        if settings["max_epochs"] == 0:\n            return _result(state, False, last_path)\n        del state\n    for epoch in range(completed + 1, settings["max_epochs"] + 1):\n        started = time.monotonic()\n        model.train()\n        optimizer.zero_grad(set_to_none=True)\n        loss_sum, count = 0.0, 0\n        for step, batch in enumerate(loader):\n            labels = batch.pop("labels").to(device)\n            batch = {key: value.to(device) for key, value in batch.items()}\n            group_start = (step // accumulation) * accumulation\n            group_documents = min(accumulation * loader.batch_size, len(train_documents) - group_start * loader.batch_size)\n            with _amp(device):\n                loss = criterion(model(**batch).float(), labels)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(f"Nonfinite training loss at epoch={epoch}, batch={step}")\n            scaler.scale(loss * len(labels) / group_documents).backward()\n            loss_sum += float(loss.detach()) * len(labels)\n            count += len(labels)\n            if (step + 1) % accumulation == 0 or step + 1 == len(loader):\n                scaler.unscale_(optimizer)\n                nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=not scaler.is_enabled())\n                old_scale = scaler.get_scale()\n                scaler.step(optimizer)\n                scaler.update()\n                if scaler.get_scale() >= old_scale:\n                    scheduler.step()\n                optimizer.zero_grad(set_to_none=True)\n            if settings["log_every"] > 0 and (step + 1) % settings["log_every"] == 0:\n                print(f"V7 seed={seed} epoch={epoch} batch={step + 1}/{len(loader)} loss={loss_sum / count:.5f}", flush=True)\n        score = selector.score(_predict(model, stop_loader, device))\n        history.append({"epoch": epoch, "phase": "full_encoder_finetune", "train_loss": loss_sum / count,\n                        **score, "minutes": (time.monotonic() - started) / 60})\n        improved = better(score, best_score)\n        if improved:\n            best_score, best_epoch, stale = score, epoch, 0\n        else:\n            stale += 1\n        done = epoch == settings["max_epochs"] or stale >= settings["patience"]\n        state = commit(epoch, done, improved)\n        print(f"V7 epoch={epoch}/{settings[\'max_epochs\']} loss={loss_sum / count:.5f} "\n              f"stop_macro_F1={score[\'stop_macro_f1\']:.6f} stop_AP={score[\'stop_macro_ap\']:.6f} "\n              f"best_epoch={best_epoch}", flush=True)\n        if done:\n            return _result(state, resumed, last_path)\n        del state\n    raise RuntimeError("Resume checkpoint marked incomplete after the configured epoch budget")\n', 'data.py': '"""Text-only turn boundaries for the independent ver7 encoder.\n\nThe shared source loader has already normalized each utterance and joined the\nturns with newlines. This representation deliberately accepts text strings,\nnever annotations, so labels and speaker metadata cannot become features.\n"""\nfrom __future__ import annotations\n\nimport unicodedata\n\n\ndef candidate_texts(texts, tokenizer):\n    """Insert the tokenizer\'s existing SEP token between nonempty turns.\n\n    No vocabulary token is added and no maximum dialogue length is imposed.\n    An empty dialogue remains empty and obtains the tokenizer\'s normal outer\n    special tokens when it is tokenized.\n    """\n    separator = getattr(tokenizer, "sep_token", None)\n    if not isinstance(separator, str) or not separator:\n        raise ValueError("The candidate tokenizer must define an existing sep_token")\n    if isinstance(texts, str):\n        raise TypeError("texts must be a sequence of dialogue strings")\n    rendered = []\n    for text in texts:\n        if not isinstance(text, str):\n            raise TypeError("Candidate inputs must be text strings, never annotations")\n        turns = [" ".join(unicodedata.normalize("NFC", turn).split()) for turn in text.split("\\n")]\n        rendered.append(f" {separator} ".join(turn for turn in turns if turn))\n    return rendered\n', 'pretrained.py': '"""Prepare one pinned public encoder; training and inference only use local files."""\nfrom __future__ import annotations\n\nimport argparse\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport tempfile\nimport uuid\n\nMODEL_ID = "monologg/koelectra-base-v3-discriminator"\nMODEL_REVISION = "68b30cd259f34a4b5aa8786392612ba2a2617fcc"\nMANIFEST_NAME = "pretrained_manifest.json"\nFORMAT = "mission3_ver7_pretrained_v1"\n\n\ndef _sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef validate_pretrained(directory) -> dict:\n    """Return provenance only after checking a complete, safe local artifact set.\n\n    This function never contacts Hugging Face and does not load model weights.\n    File hashes are returned under ``files[name][\'sha256\']`` for run identities.\n    """\n    root = Path(directory).expanduser().resolve()\n    marker = root / MANIFEST_NAME\n    if not marker.is_file() or marker.is_symlink():\n        raise ValueError(f"사전학습 모델 준비가 완료되지 않았습니다: {marker}")\n    try:\n        metadata = json.loads(marker.read_text(encoding="utf-8"))\n    except (OSError, ValueError) as error:\n        raise ValueError(f"잘못된 사전학습 모델 manifest: {marker}") from error\n    if (metadata.get("format") != FORMAT or metadata.get("complete") is not True\n            or not isinstance(metadata.get("model_id"), str)\n            or not re.fullmatch(r"[0-9a-f]{40}", str(metadata.get("revision", "")))):\n        raise ValueError("사전학습 모델 manifest 형식/고정 revision이 올바르지 않습니다.")\n    files = metadata.get("files")\n    if not isinstance(files, dict) or not files:\n        raise ValueError("사전학습 모델 파일 목록이 없습니다.")\n    required = {"config.json", "tokenizer_config.json", "tokenizer.json"}\n    if not required <= files.keys() or not ("model.safetensors" in files\n                                           or "model.safetensors.index.json" in files):\n        raise ValueError("encoder config/safetensors/fast tokenizer 파일이 빠졌습니다.")\n    actual = {p.relative_to(root).as_posix() for p in root.rglob("*") if p.is_file()}\n    if actual != set(files) | {MANIFEST_NAME}:\n        raise ValueError("사전학습 모델의 실제 파일 목록과 manifest가 다릅니다.")\n    for name, expected in files.items():\n        relative = PurePosixPath(name)\n        if (relative.is_absolute() or ".." in relative.parts or "\\\\" in name\n                or relative.as_posix() != name or name == MANIFEST_NAME):\n            raise ValueError(f"안전하지 않은 사전학습 파일 경로: {name}")\n        path = root / name\n        if (path.is_symlink() or any(parent.is_symlink() for parent in path.parents if parent != root)\n                or not path.is_file() or not path.resolve().is_relative_to(root)):\n            raise ValueError(f"정상적인 로컬 사전학습 파일이 아닙니다: {name}")\n        if (not isinstance(expected, dict) or path.stat().st_size != expected.get("size")\n                or _sha256(path) != expected.get("sha256")):\n            raise ValueError(f"사전학습 파일 무결성 검사 실패: {name}")\n    return metadata\n\n\ndef prepare_pretrained(destination, model_id=MODEL_ID, revision=MODEL_REVISION) -> dict:\n    """Download a pinned encoder/tokenizer once and atomically publish local files.\n\n    Only this explicit preparation entry point may use the network. It receives no\n    competition texts or labels. A valid completed directory is reused offline.\n    A failed preparation leaves the existing destination untouched.\n    """\n    if not re.fullmatch(r"[0-9a-f]{40}", str(revision)):\n        raise ValueError("revision은 변경되지 않는 40자리 Hugging Face commit SHA여야 합니다.")\n    raw_destination = Path(destination).expanduser()\n    if raw_destination.is_symlink():\n        raise ValueError("사전학습 모델 저장 경로에 심볼릭 링크를 사용할 수 없습니다.")\n    root = raw_destination.resolve()\n    if root.exists():\n        try:\n            metadata = validate_pretrained(root)\n        except ValueError:\n            # Refuse to replace unrelated directories; our own corrupt bundle may be repaired.\n            marker = root / MANIFEST_NAME\n            try:\n                previous = json.loads(marker.read_text(encoding="utf-8"))\n            except (OSError, ValueError):\n                previous = {}\n            if previous.get("format") != FORMAT:\n                raise ValueError(f"완료 manifest가 없는 기존 폴더입니다. 비어 있는 새 경로를 사용하세요: {root}")\n        else:\n            if metadata["model_id"] != model_id or metadata["revision"] != revision:\n                raise ValueError("기존 사전학습 모델과 요청한 모델/revision이 다릅니다. 새 경로를 사용하세요.")\n            print(f"검증된 사전학습 모델 재사용: {root}", flush=True)\n            return metadata\n    root.parent.mkdir(parents=True, exist_ok=True)\n    staging = Path(tempfile.mkdtemp(prefix="." + root.name + ".preparing-", dir=root.parent))\n    backup = None\n    try:\n        from transformers import AutoModel, AutoTokenizer\n\n        print(f"공개 사전학습 encoder 다운로드: {model_id}@{revision}", flush=True)\n        options = {"revision": revision, "local_files_only": False, "trust_remote_code": False}\n        tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True, **options)\n        # The pinned upstream commit contains PyTorch weights, not safetensors.\n        # torch>=2.6 uses the restricted weights-only loader; no remote code runs.\n        model = AutoModel.from_pretrained(model_id, use_safetensors=False, **options)\n        tokenizer.save_pretrained(staging)\n        model.save_pretrained(staging, safe_serialization=True)\n        del model, tokenizer\n        # Prove that the published form is self-contained and works offline.\n        check_model = AutoModel.from_pretrained(staging, local_files_only=True, trust_remote_code=False,\n                                               use_safetensors=True)\n        check_tokenizer = AutoTokenizer.from_pretrained(staging, local_files_only=True,\n                                                       trust_remote_code=False, use_fast=True)\n        if not check_tokenizer.is_fast:\n            raise ValueError("window 생성에 사용할 fast tokenizer가 필요합니다.")\n        del check_model, check_tokenizer\n        metadata = {\n            "format": FORMAT, "complete": True, "model_id": model_id, "revision": revision,\n            "prepared_at_utc": datetime.now(timezone.utc).isoformat(),\n            "files": {path.relative_to(staging).as_posix(): {"size": path.stat().st_size, "sha256": _sha256(path)}\n                      for path in sorted(staging.rglob("*")) if path.is_file()},\n        }\n        (staging / MANIFEST_NAME).write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + "\\n",\n                                           encoding="utf-8")\n        validate_pretrained(staging)\n        if root.exists():\n            backup = root.with_name("." + root.name + ".previous-" + uuid.uuid4().hex)\n            os.replace(root, backup)\n        try:\n            os.replace(staging, root)\n        except BaseException:\n            if backup is not None:\n                os.replace(backup, root)\n                backup = None\n            raise\n        if backup is not None:\n            shutil.rmtree(backup)\n            backup = None\n        print(f"사전학습 모델 준비 완료: {root}", flush=True)\n        return metadata\n    finally:\n        if staging.exists():\n            shutil.rmtree(staging)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--destination", required=True)\n    parser.add_argument("--model_id", default=MODEL_ID)\n    parser.add_argument("--revision", default=MODEL_REVISION)\n    args = parser.parse_args()\n    prepare_pretrained(args.destination, model_id=args.model_id, revision=args.revision)\n\n\nif __name__ == "__main__":\n    main()\n', 'selection.py': '"""Grouped threshold cross-fitting on the original fold\'s stopping split.\n\nThis is model/epoch selection on Training stop. It is not model-level OOF and\nits repeatedly selected score is not an unbiased final performance estimate.\nNo official Validation or model-calibration rows belong here.\n"""\nfrom __future__ import annotations\n\nimport numpy as np\nfrom sklearn.metrics import average_precision_score, f1_score\n\nif __package__:\n    from .ver4_runtime.calibration import _best_threshold, _grouped_splits\nelse:\n    from ver4_runtime.calibration import _best_threshold, _grouped_splits\n\n\nclass StopSelector:\n    def __init__(self, targets, groups, seed=44, n_splits=3):\n        self.targets = np.asarray(targets, dtype=np.float32)\n        if self.targets.ndim != 2 or not len(self.targets) or not np.isin(self.targets, [0, 1]).all():\n            raise ValueError("Stop targets must be a nonempty binary matrix")\n        self.groups = np.asarray(groups)\n        self.folds = list(_grouped_splits(self.targets, self.groups, seed, n_splits))\n\n    def score(self, probabilities):\n        probabilities = np.asarray(probabilities, dtype=np.float64)\n        if probabilities.shape != self.targets.shape or not np.isfinite(probabilities).all():\n            raise ValueError("Stop probabilities must be finite and match targets")\n        if np.any((probabilities < 0) | (probabilities > 1)):\n            raise ValueError("Stop probabilities must lie in [0, 1]")\n        predictions = np.zeros_like(self.targets, dtype=np.int8)\n        seen = np.zeros(len(self.targets), dtype=np.int8)\n        thresholds = []\n        for fit, score in self.folds:\n            threshold = [_best_threshold(self.targets[fit, label], probabilities[fit, label])[0]\n                         for label in range(self.targets.shape[1])]\n            predictions[score] = probabilities[score] >= np.asarray(threshold)\n            seen[score] += 1\n            thresholds.append(threshold)\n        if not np.all(seen == 1):\n            raise RuntimeError("Every stop row must be scored once")\n        present = self.targets.sum(0) > 0\n        ap = float(average_precision_score(self.targets[:, present], probabilities[:, present], average="macro")) if present.any() else 0.0\n        return {"stop_macro_f1": float(f1_score(self.targets, predictions, average="macro", zero_division=0)),\n                "stop_macro_ap": ap, "stop_labels_with_positives": int(present.sum()),\n                "threshold_crossfit_splits": len(self.folds), "thresholds_by_fold": thresholds}\n\n\ndef better(score, best, tolerance=1e-6):\n    """F1 selects; AP is only a deterministic secondary criterion for F1 ties."""\n    if best is None:\n        return True\n    delta = score["stop_macro_f1"] - best["stop_macro_f1"]\n    return delta > tolerance or (abs(delta) <= tolerance and score["stop_macro_ap"] > best["stop_macro_ap"] + tolerance)\n', 'calibration.py': '"""Training-only selection between ver4 and two bounded three-way mixtures."""\nfrom __future__ import annotations\n\nimport numpy as np\nfrom sklearn.metrics import f1_score, precision_recall_fscore_support\n\nif __package__:\n    from .ver4_runtime.calibration import (\n        WEIGHT_GRID, _best_threshold, _grouped_splits, _inputs, _probabilities,\n        apply_calibrator as _apply_baseline, fit_calibrator as _fit_baseline,\n    )\n    from .ver4_runtime.data import TARGET_SYMPTOMS\nelse:\n    from ver4_runtime.calibration import (\n        WEIGHT_GRID, _best_threshold, _grouped_splits, _inputs, _probabilities,\n        apply_calibrator as _apply_baseline, fit_calibrator as _fit_baseline,\n    )\n    from ver4_runtime.data import TARGET_SYMPTOMS\n\nCANDIDATE_GRID = (0.0, 0.25, 0.5, 0.75, 1.0)\nLOGIT_EPSILON = 1e-6\n\n\ndef _blend_space(value):\n    if value not in ("raw", "logit"):\n        raise ValueError("blend_space must be \'raw\' or \'logit\'")\n    return value\n\n\ndef _logits(probabilities):\n    probabilities = np.clip(np.asarray(probabilities, dtype=np.float64),\n                            LOGIT_EPSILON, 1.0 - LOGIT_EPSILON)\n    return np.log(probabilities) - np.log1p(-probabilities)\n\n\ndef _sigmoid(logits):\n    # Convex mixtures of clipped logits stay within finite, safe exp bounds.\n    return 1.0 / (1.0 + np.exp(-logits))\n\n\ndef _validated_inputs(y_true, base_neural, sparse, candidate):\n    truth, base, sparse = _inputs(y_true, base_neural, sparse)\n    candidate = _probabilities(candidate, "candidate")\n    if candidate.shape != base.shape:\n        raise ValueError("candidate and base_neural must have identical shapes")\n    return truth, base, sparse, candidate\n\n\ndef _with_zero_candidate(calibrator):\n    return {**calibrator, "candidate_weights": [0.0] * len(TARGET_SYMPTOMS),\n            "blend_space": "raw"}\n\n\ndef _fit_mixture(y_true, base_neural, sparse, candidate, blend_space="raw"):\n    """Exact F1 selection per label; equal scores prefer zero candidate weight."""\n    blend_space = _blend_space(blend_space)\n    truth, base, sparse, candidate = _validated_inputs(y_true, base_neural, sparse, candidate)\n    if blend_space == "logit":\n        base, sparse, candidate = map(_logits, (base, sparse, candidate))\n    alphas, betas, thresholds = [], [], []\n    for column in range(len(TARGET_SYMPTOMS)):\n        best = None\n        for alpha in WEIGHT_GRID:\n            baseline = alpha * base[:, column] + (1.0 - alpha) * sparse[:, column]\n            for beta in CANDIDATE_GRID:\n                probabilities = (1.0 - beta) * baseline + beta * candidate[:, column]\n                if blend_space == "logit":\n                    probabilities = _sigmoid(probabilities)\n                threshold, score = _best_threshold(truth[:, column], probabilities)\n                # Keep ver4\'s alpha tie convention after preferring smaller beta.\n                choice = (score, -beta, -abs(alpha - 0.5), alpha, threshold)\n                if best is None or choice > best:\n                    best = choice\n        _, negative_beta, _, alpha, threshold = best\n        alphas.append(float(alpha))\n        betas.append(float(-negative_beta))\n        thresholds.append(float(threshold))\n    return {"neural_weights": alphas, "candidate_weights": betas,\n            "thresholds": thresholds, "blend_space": blend_space}\n\n\ndef apply_calibrator(base_neural, sparse, candidate, calibrator):\n    """Apply the stored space; older manifests default to the exact raw blend."""\n    blend_space = _blend_space(calibrator.get("blend_space", "raw"))\n    # Reuse ver4\'s shape, range, alpha and threshold validation in both spaces.\n    baseline = _apply_baseline(base_neural, sparse, calibrator)\n    betas = np.asarray(calibrator["candidate_weights"], dtype=np.float64)\n    if betas.shape != (len(TARGET_SYMPTOMS),) or not np.isfinite(betas).all():\n        raise ValueError("candidate_weights must contain one finite weight per symptom")\n    if np.any((betas < 0) | (betas > 1)):\n        raise ValueError("candidate_weights must be in [0, 1]")\n    if blend_space == "logit":\n        alphas = np.asarray(calibrator["neural_weights"], dtype=np.float64)\n        baseline = alphas * _logits(base_neural) + (1.0 - alphas) * _logits(sparse)\n    if candidate is None:\n        if np.any(betas != 0):\n            raise ValueError("candidate probabilities are required for nonzero candidate_weights")\n        return _sigmoid(baseline) if blend_space == "logit" else baseline\n    candidate = _probabilities(candidate, "candidate")\n    if candidate.shape != baseline.shape:\n        raise ValueError("candidate and base_neural must have identical shapes")\n    if blend_space == "logit":\n        candidate = _logits(candidate)\n    mixture = (1.0 - betas) * baseline + betas * candidate\n    return _sigmoid(mixture) if blend_space == "logit" else mixture\n\n\ndef _metrics(truth, predictions):\n    precision, recall, f1, support = precision_recall_fscore_support(\n        truth, predictions, average=None, zero_division=0,\n    )\n    return {\n        symptom: {"precision": float(p), "recall": float(r), "f1": float(f), "support": int(s)}\n        for symptom, p, r, f, s in zip(TARGET_SYMPTOMS, precision, recall, f1, support)\n    }\n\n\ndef fit_candidate_only_crossfit(y_true, candidate, groups, seed=44, n_splits=5):\n    """Diagnose a standalone candidate using grouped Training calibration only.\n\n    The candidate model must exclude all supplied samples from model fitting and\n    epoch selection. Each scored fold uses thresholds fitted on the other\n    groups; the returned thresholds are then fitted on all Training calibration\n    rows. Never supply official Validation. This diagnostic does not participate\n    in the baseline/mixture selection procedure or its adoption guard.\n    """\n    candidate = _probabilities(candidate, "candidate")\n    truth, candidate, _ = _inputs(y_true, candidate, candidate)\n\n    def fit_thresholds(rows):\n        return [\n            float(_best_threshold(truth[rows, column], candidate[rows, column])[0])\n            for column in range(len(TARGET_SYMPTOMS))\n        ]\n\n    predictions = np.zeros_like(truth)\n    seen = np.zeros(len(truth), dtype=np.int64)\n    fold_count = 0\n    for train, test in _grouped_splits(truth, groups, seed, n_splits):\n        thresholds = np.asarray(fit_thresholds(train), dtype=np.float64)\n        predictions[test] = candidate[test] >= thresholds\n        np.add.at(seen, test, 1)\n        fold_count += 1\n    if not np.all(seen == 1):\n        raise RuntimeError("every calibration sample must be scored exactly once")\n\n    calibrator = {"thresholds": fit_thresholds(np.arange(len(truth)))}\n    fitted = candidate >= np.asarray(calibrator["thresholds"], dtype=np.float64)\n    report = {\n        "candidate_standalone_crossfit_macro_f1": float(\n            f1_score(truth, predictions, average="macro", zero_division=0)),\n        "candidate_standalone_fit_macro_f1": float(\n            f1_score(truth, fitted, average="macro", zero_division=0)),\n        "candidate_standalone_per_symptom_crossfit": _metrics(truth, predictions),\n        "candidate_standalone_crossfit_splits": fold_count,\n    }\n    return calibrator, report\n\n\ndef calibrate_crossfit(y_true, base_neural, sparse, candidate, groups, seed=44, n_splits=5,\n                       min_gain=0.002, min_improved_folds=3):\n    """Tune on calibration-fold training rows, compare on excluded groups.\n\n    All supplied probabilities must originate from base/candidate models that\n    excluded this entire Training calibration set from model fitting and epoch\n    selection. These arrays must never contain official Validation samples.\n    Compare only the original baseline and fixed raw/logit mixture families.\n    The global procedure choice uses the resulting cross-fit scores, so its\n    selected score is a selection estimate, not an unbiased performance guarantee.\n    """\n    if not np.isfinite(min_gain) or min_gain < 0:\n        raise ValueError("min_gain must be finite and nonnegative")\n    if type(min_improved_folds) is not int or min_improved_folds < 1:\n        raise ValueError("min_improved_folds must be a positive integer")\n    truth, base, sparse, candidate = _validated_inputs(y_true, base_neural, sparse, candidate)\n    baseline_predictions = np.zeros_like(truth)\n    mixture_predictions = {space: np.zeros_like(truth) for space in ("raw", "logit")}\n    seen = np.zeros(len(truth), dtype=np.int8)\n    fold_counts = []\n    fold_gains = {space: [] for space in ("raw", "logit")}\n    for train, test in _grouped_splits(truth, groups, seed, n_splits):\n        baseline_calibrator = _fit_baseline(truth[train], base[train], sparse[train])\n        baseline_predictions[test] = (\n            _apply_baseline(base[test], sparse[test], baseline_calibrator)\n            >= np.asarray(baseline_calibrator["thresholds"])\n        )\n        fold_baseline_score = f1_score(truth[test], baseline_predictions[test], average="macro", zero_division=0)\n        for space, predictions in mixture_predictions.items():\n            mixture_calibrator = _fit_mixture(\n                truth[train], base[train], sparse[train], candidate[train], blend_space=space,\n            )\n            predictions[test] = (\n                apply_calibrator(base[test], sparse[test], candidate[test], mixture_calibrator)\n                >= np.asarray(mixture_calibrator["thresholds"])\n            )\n            fold_gains[space].append(float(f1_score(truth[test], predictions[test], average="macro", zero_division=0)\n                                          - fold_baseline_score))\n        seen[test] += 1\n        fold_counts.append({"fit_samples": int(len(train)), "score_samples": int(len(test))})\n    if not np.all(seen == 1):\n        raise RuntimeError("every calibration sample must be scored exactly once")\n    baseline_score = float(f1_score(truth, baseline_predictions, average="macro", zero_division=0))\n    mixture_scores = {\n        space: float(f1_score(truth, predictions, average="macro", zero_division=0))\n        for space, predictions in mixture_predictions.items()\n    }\n    # A single global family choice avoids per-symptom selection on scored folds.\n    candidate_space = "logit" if mixture_scores["logit"] > mixture_scores["raw"] else "raw"\n    candidate_score = mixture_scores[candidate_space]\n    candidate_predictions = mixture_predictions[candidate_space]\n    baseline_final = _with_zero_candidate(_fit_baseline(truth, base, sparse))\n    gain = candidate_score - baseline_score\n    required_folds = min(min_improved_folds, len(fold_counts))\n    improved_folds = sum(value > 0 for value in fold_gains[candidate_space])\n    material = candidate_score > baseline_score and gain >= min_gain\n    consistent = improved_folds >= required_folds\n    use_candidate = material and consistent\n    selected = (_fit_mixture(truth, base, sparse, candidate, blend_space=candidate_space)\n                if use_candidate else baseline_final)\n    selected_predictions = candidate_predictions if use_candidate else baseline_predictions\n    fitted = apply_calibrator(base, sparse, candidate, selected) >= np.asarray(selected["thresholds"])\n    report = {\n        "score_scope": "Training-only calibration. Procedure selection reuses cross-fit scores; selected score is a selection estimate, not an unbiased guarantee or official Validation.",\n        "baseline_crossfit_macro_f1": baseline_score,\n        "candidate_crossfit_macro_f1": candidate_score,\n        "candidate_raw_crossfit_macro_f1": mixture_scores["raw"],\n        "candidate_logit_crossfit_macro_f1": mixture_scores["logit"],\n        "candidate_blend_space": candidate_space,\n        "selected_blend_space": selected["blend_space"],\n        "selected_crossfit_macro_f1": candidate_score if use_candidate else baseline_score,\n        "selected_fit_macro_f1": float(f1_score(truth, fitted, average="macro", zero_division=0)),\n        "selected_name": "candidate_mixture" if use_candidate else "ver4_baseline",\n        "fallback_reason": (None if use_candidate else\n                            "candidate_gain_below_minimum" if not material else "candidate_gain_not_consistent_across_folds"),\n        "selection_gain": float(gain), "minimum_gain": float(min_gain),\n        "candidate_improved_folds": improved_folds, "minimum_improved_folds": required_folds,\n        "candidate_fold_gains": fold_gains[candidate_space], "all_candidate_fold_gains": fold_gains,\n        "per_symptom_crossfit": {\n            "baseline": _metrics(truth, baseline_predictions),\n            "candidate": _metrics(truth, candidate_predictions),\n            "candidate_raw": _metrics(truth, mixture_predictions["raw"]),\n            "candidate_logit": _metrics(truth, mixture_predictions["logit"]),\n            "selected": _metrics(truth, selected_predictions),\n        },\n        "calibration_samples": int(len(truth)),\n        "calibration_groups": int(len(np.unique(groups))),\n        "crossfit_splits": len(fold_counts),\n        "crossfit_fold_counts": fold_counts,\n        "selection_rule": "Compare fixed raw/logit families globally; ties prefer raw. Require strictly positive aggregate gain at least minimum_gain and improvements on minimum_improved_folds; otherwise use ver4. Training-only selection guard, not a statistical significance test or Validation guarantee.",\n    }\n    return selected, report\n', 'prediction_cache.py': '"""Reuse raw inference probabilities across experiments without reading labels.\n\nKeys bind ordered inputs, weights, tokenizer, inference code and execution\nsettings. Calibration/thresholds are deliberately excluded: they are applied\nafter loading raw probabilities. A missing or damaged cache is recomputed.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path\nimport tempfile\nfrom zipfile import BadZipFile\n\nimport numpy as np\n\n\ndef _file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _json(value):\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))\n\n\nclass PredictionCache:\n    def __init__(self, directory, paths, texts, tokenizer_dir, labels, execution):\n        self.directory = Path(directory).expanduser().resolve()\n        self.shape = (len(paths), len(labels))\n        if len(paths) != len(texts):\n            raise ValueError("Cache paths/texts length mismatch")\n        tokenizer_dir = Path(tokenizer_dir)\n        self.context = {\n            "format": "mission3_raw_probabilities_v1",\n            "inputs": hashlib.sha256(_json([\n                [Path(path).name, text] for path, text in zip(paths, texts)\n            ]).encode("utf-8")).hexdigest(),\n            "labels": list(labels), "execution": execution,\n            "tokenizer": {p.relative_to(tokenizer_dir).as_posix(): _file_hash(p)\n                          for p in sorted(tokenizer_dir.rglob("*")) if p.is_file()},\n        }\n\n    def _valid(self, values):\n        return (values.shape == self.shape and np.isfinite(values).all()\n                and np.all((values >= 0) & (values <= 1)))\n\n    def get(self, kind, checkpoint, config, sources, compute):\n        # Do not bind absolute paths: a copied identical bundle can reuse results.\n        identity = _json({\n            **self.context, "kind": kind, "weights": _file_hash(checkpoint),\n            "settings": {key: config[key] for key in\n                         ("device", "max_length", "stride", "window_batch_size", "inference_batch_size")},\n            "code": [_file_hash(path) for path in sources],\n        })\n        key = hashlib.sha256(identity.encode("utf-8")).hexdigest()\n        path = self.directory / (key + ".npz")\n        if path.is_file():\n            try:\n                with np.load(path, allow_pickle=False) as stored:\n                    values = np.array(stored["probabilities"], dtype=np.float64)\n                    checksum = hashlib.sha256(values.tobytes()).hexdigest()\n                    if (stored["identity"].item() == identity and self._valid(values)\n                            and stored["checksum"].item() == checksum):\n                        print(f"예측 캐시 재사용: {kind} / {Path(checkpoint).name}", flush=True)\n                        return values\n            except (OSError, ValueError, KeyError, EOFError, BadZipFile):\n                pass\n            print(f"예측 캐시 손상: {kind}, 다시 계산합니다.", flush=True)\n        values = np.asarray(compute(), dtype=np.float64)\n        if not self._valid(values):\n            raise ValueError("Invalid raw probabilities; cache was not written")\n        self.directory.mkdir(parents=True, exist_ok=True)\n        # Unique temporary files keep interrupted/concurrent writes out of reads.\n        temporary = None\n        try:\n            with tempfile.NamedTemporaryFile(dir=self.directory, suffix=".tmp", delete=False) as stream:\n                temporary = Path(stream.name)\n                np.savez_compressed(stream, identity=np.asarray(identity), probabilities=values,\n                                    checksum=np.asarray(hashlib.sha256(values.tobytes()).hexdigest()))\n            temporary.replace(path)\n        finally:\n            if temporary is not None:\n                temporary.unlink(missing_ok=True)\n        print(f"예측 캐시 저장: {kind} / {Path(checkpoint).name}", flush=True)\n        return values\n', 'pipeline.py': '"""Reuse ver4; fit one complementary model without touching official Validation."""\nfrom __future__ import annotations\n\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport torch\nfrom transformers import AutoTokenizer\n\nif __package__:\n    from . import calibration, data, neural, pretrained\n    from .selection import StopSelector\n    from .prediction_cache import PredictionCache\n    from .ver4_runtime import pipeline as base_pipeline\n    from .ver4_runtime import neural as base_neural\n    from .ver4_runtime.data import TARGET_SYMPTOMS, load_dataset, text_group, write_json, write_predictions\nelse:\n    import calibration\n    import data\n    import pretrained\n    from selection import StopSelector\n    import neural\n    from prediction_cache import PredictionCache\n    from ver4_runtime import pipeline as base_pipeline\n    from ver4_runtime import neural as base_neural\n    from ver4_runtime.data import TARGET_SYMPTOMS, load_dataset, text_group, write_json, write_predictions\n\n\nDEFAULT_CONFIG = {\n    **neural.DEFAULTS, "seed": 42, "init_fold": 1, "require_cuda": True,\n    "bundle_base": False,\n    "min_calibration_gain": 0.002, "min_calibration_improved_folds": 3,\n}\nRUNTIME_FILES = (\n    "__init__.py", "pipeline.py", "neural.py", "data.py", "selection.py", "pretrained.py",\n    "calibration.py", "prediction_cache.py", "train.py", "inference.py",\n    "bootstrap.py", "requirements.txt", "README.md", "RESEARCH.md",\n)\n\n\ndef _sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _atomic_torch_save(path, value):\n    path = Path(path)\n    temporary = path.with_name(path.name + ".tmp")\n    torch.save(value, temporary)\n    temporary.replace(path)\n\n\ndef _release():\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\ndef _versions():\n    return {name: importlib.metadata.version(name) for name in\n            ("torch", "transformers", "numpy", "scikit-learn", "iterative-stratification")}\n\n\ndef _check_versions(base_manifest):\n    current = _versions()\n    # sklearn objects in sparse.joblib require the original compatible environment.\n    for name in ("transformers", "numpy", "scikit-learn"):\n        saved = base_manifest.get("versions", {}).get(name)\n        if saved and current[name] != saved:\n            raise RuntimeError(f"{name}: ver4 저장 버전={saved}, 현재={current[name]}. requirements.txt를 설치하세요.")\n    return current\n\n\ndef _read_base(train_label_dir, base_dir, init_fold):\n    """Verify the exact original training run before trusting cached predictions."""\n    checkpoint = base_dir / "mission3_ver4_model.ckpt"\n    manifest = base_pipeline.load_manifest(checkpoint)\n    _check_versions(manifest)\n    print("기존 ver4 확인 완료. 서울 Training JSON을 읽습니다...", flush=True)\n    paths, texts, targets = load_dataset(train_label_dir, include_targets=True)\n    print(f"Training {len(paths):,}개 로드 완료. 원래 데이터·분할·보정 캐시를 검증합니다...", flush=True)\n    fingerprint = base_pipeline._fingerprint(paths, texts, targets, manifest["config"])\n    if fingerprint != manifest.get("signature"):\n        raise ValueError("ver4와 Training 데이터/전처리/코드가 다릅니다. 원래 Training과 ver4 결과 묶음을 사용하세요.")\n    splits = json.loads((base_dir / "splits.json").read_text(encoding="utf-8"))\n    lookup = {p.name: i for i, p in enumerate(paths)}\n    all_names = set(lookup)\n    group_ids = np.asarray([text_group(text) for text in texts])\n\n    def indices(names, purpose):\n        if not names or len(names) != len(set(names)) or not set(names) <= all_names:\n            raise ValueError(f"Invalid saved {purpose} filenames")\n        return np.asarray([lookup[name] for name in names], dtype=np.int64)\n\n    cal = indices(splits["calibration"], "calibration")\n    if len(splits["folds"]) != len(manifest["folds"]) or not 1 <= init_fold <= len(splits["folds"]):\n        raise ValueError("init_fold / saved folds mismatch")\n    fold_indices = []\n    for number, fold in enumerate(splits["folds"], 1):\n        fit, stop = indices(fold["fit"], "fit"), indices(fold["stop"], "stop")\n        sets = [set(fit), set(stop), set(cal)]\n        groups = [set(group_ids[x]) for x in (fit, stop, cal)]\n        if any(sets[i] & sets[j] or groups[i] & groups[j] for i, j in ((0, 1), (0, 2), (1, 2))):\n            raise ValueError(f"fold {number}: sample or text-group leakage")\n        if set.union(*sets) != set(range(len(paths))):\n            raise ValueError(f"fold {number}: saved partitions do not cover Training")\n        fold_indices.append((fit, stop))\n    # Verify the stored fold identities against the ORIGINAL deterministic run.\n    # Preserve the old models\' exclusions and the comparable single-fold stop.\n    _, expected_cal, expected_folds, _ = base_pipeline.make_splits(\n        texts, targets, manifest["config"]["calibration_size"],\n        manifest["config"]["n_folds"], manifest["config"]["seed"],\n    )\n    if not np.array_equal(cal, expected_cal) or any(\n        not np.array_equal(actual, expected)\n        for pair, reference in zip(fold_indices, expected_folds)\n        for actual, expected in zip(pair, reference)\n    ):\n        raise ValueError("Saved fold identity differs from the original ver4 run; base reuse would leak.")\n    for number, name in enumerate(manifest["folds"], 1):\n        saved = torch.load(base_dir / name, map_location="cpu", weights_only=True, mmap=True)\n        if (saved.get("format_version") != 1\n                or saved.get("seed") != manifest["config"]["seed"] + number * 1009\n                or saved.get("architecture", {}).get("num_labels") != len(TARGET_SYMPTOMS)):\n            raise ValueError(f"Saved checkpoint fold identity mismatch: {name}")\n        del saved\n    with np.load(base_dir / "calibration_predictions.npz", allow_pickle=False) as cache:\n        names = cache["filenames"].tolist()\n        if len(set(names)) != len(names) or set(names) != set(splits["calibration"]):\n            raise ValueError("ver4 calibration cache filenames mismatch")\n        order = {name: i for i, name in enumerate(names)}\n        aligned = [order[paths[i].name] for i in cal]\n        raw_neural = np.array(cache["neural"][aligned], dtype=np.float64)\n        raw_sparse = np.array(cache["sparse"][aligned], dtype=np.float64)\n        if not np.array_equal(cache["targets"][aligned], targets[cal]):\n            raise ValueError("ver4 calibration cache targets mismatch")\n    for values in (raw_neural, raw_sparse):\n        if values.shape != (len(cal), len(TARGET_SYMPTOMS)) or not np.isfinite(values).all() or np.any((values < 0) | (values > 1)):\n            raise ValueError("Invalid ver4 raw calibration probabilities")\n    fit, stop = fold_indices[init_fold - 1]\n    return manifest, paths, texts, targets, fit, stop, cal, group_ids, raw_neural, raw_sparse\n\n\ndef _run_signature(config, base_manifest, base_dir, pretrained_metadata):\n    source_dir = Path(__file__).parent\n    code = hashlib.sha256()\n    own_code = [source_dir / name for name in RUNTIME_FILES if name.endswith(".py")]\n    for path in sorted([*own_code, *source_dir.joinpath("ver4_runtime").glob("*.py")]):\n        code.update(str(path.relative_to(source_dir)).encode())\n        code.update(path.read_bytes())\n    assets = [base_dir / name for name in [*base_manifest["folds"], base_manifest["sparse"], "mission3_ver4_model.ckpt"]]\n    assets += [path for path in (base_dir / base_manifest["tokenizer"]).rglob("*") if path.is_file()]\n    asset_hashes = {}\n    for path in sorted(assets):\n        name = str(path.relative_to(base_dir))\n        print("기본 모델 무결성 확인:", name, flush=True)\n        asset_hashes[name] = _sha256(path)\n    value = {\n        "config": config, "base_signature": base_manifest["signature"],\n        "public_pretrained": {key: pretrained_metadata[key] for key in ("model_id", "revision", "files")},\n        "base_artifact_sha256": asset_hashes,\n        "calibration_cache_sha256": _sha256(base_dir / "calibration_predictions.npz"),\n        "splits_sha256": _sha256(base_dir / "splits.json"), "code_sha256": code.hexdigest(),\n        "versions": _versions(),\n    }\n    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest(), value\n\n\ndef _copy_file(source, target, expected_hash):\n    """An interrupted copy never appears under its final filename."""\n    target.parent.mkdir(parents=True, exist_ok=True)\n    if target.exists():\n        if target.stat().st_size != source.stat().st_size or _sha256(target) != expected_hash:\n            raise ValueError(f"기존 묶음 파일 내용이 다릅니다: {target}")\n        return\n    temporary = target.with_name(target.name + ".tmp")\n    digest = hashlib.sha256()\n    with source.open("rb") as reader, temporary.open("wb") as writer:\n        for chunk in iter(lambda: reader.read(8 * 1024 * 1024), b""):\n            writer.write(chunk)\n            digest.update(chunk)\n    if digest.hexdigest() != expected_hash:\n        temporary.unlink()\n        raise ValueError(f"복사 중 기본 모델 파일이 변경되었습니다: {source}")\n    temporary.replace(target)\n\n\ndef _export_base(base_dir, output_dir, manifest, asset_hashes):\n    destination = output_dir / "base"\n    destination.mkdir(exist_ok=True)\n    # Copy to make the resulting ver7 folder portable without its old sibling.\n    for name in [*manifest["folds"], manifest["sparse"], "mission3_ver4_model.ckpt"]:\n        print("Bundle:", name, flush=True)\n        _copy_file(base_dir / name, destination / name, asset_hashes[name])\n    tokenizer_dir = base_dir / manifest["tokenizer"]\n    for path in tokenizer_dir.rglob("*"):\n        if path.is_file():\n            _copy_file(path, destination / manifest["tokenizer"] / path.relative_to(tokenizer_dir),\n                       asset_hashes[str(path.relative_to(base_dir))])\n    copied = base_pipeline.load_manifest(destination / "mission3_ver4_model.ckpt")\n    if copied["signature"] != manifest["signature"]:\n        raise ValueError("Portable base bundle signature mismatch")\n\n\ndef _export_code(output_dir):\n    source = Path(__file__).parent\n    submission = output_dir / "submission"\n    submission.mkdir(exist_ok=True)\n    for name in RUNTIME_FILES:\n        if (source / name).is_file():\n            shutil.copyfile(source / name, submission / name)\n    for path in source.joinpath("ver4_runtime").iterdir():\n        if path.is_file():\n            destination = submission / "ver4_runtime" / path.name\n            destination.parent.mkdir(exist_ok=True)\n            shutil.copyfile(path, destination)\n\n\ndef _checked_probabilities(values, rows):\n    values = np.asarray(values, dtype=np.float64)\n    if (values.shape != (rows, len(TARGET_SYMPTOMS)) or not np.isfinite(values).all()\n            or np.any((values < 0) | (values > 1))):\n        raise ValueError("Invalid cached/model probabilities")\n    return values\n\n\ndef _cached_predictions(path, signature, filenames, weights_digest, compute):\n    """Bind a Training diagnostic cache to its exact model and held-out rows."""\n    if path.is_file():\n        with np.load(path, allow_pickle=False) as cache:\n            if (cache["signature"].item() != signature or cache["filenames"].tolist() != filenames\n                    or cache["weights_sha256"].item() != weights_digest):\n                raise ValueError("Training prediction cache mismatch")\n            values = np.array(cache["probabilities"])\n    else:\n        values = _checked_probabilities(compute(), len(filenames))\n        temporary = path.with_name(path.name + ".tmp")\n        with temporary.open("wb") as stream:\n            np.savez_compressed(stream, probabilities=values, filenames=np.asarray(filenames),\n                                signature=np.asarray(signature), weights_sha256=np.asarray(weights_digest))\n        temporary.replace(path)\n    return _checked_probabilities(values, len(filenames))\n\n\ndef _tokenizer_hashes(directory):\n    directory = Path(directory)\n    return {p.relative_to(directory).as_posix(): _sha256(p)\n            for p in sorted(directory.rglob("*")) if p.is_file()}\n\n\ndef train_pipeline(train_label_dir, base_dir, output_dir, config=None, pretrained_dir=None):\n    """Fit one fresh public encoder on the original Training fold, then calibrate.\n\n    Official Validation is deliberately not an argument. The old task encoder is\n    used only for the baseline comparison/ensemble, never as initialization.\n    """\n    started = time.monotonic()\n    supplied = config or {}\n    unknown = set(supplied) - set(DEFAULT_CONFIG)\n    if unknown:\n        raise ValueError(f"Unknown config keys: {sorted(unknown)}")\n    config = {**DEFAULT_CONFIG, **supplied}\n    if not isinstance(config["bundle_base"], bool):\n        raise ValueError("bundle_base must be true or false")\n    if not np.isfinite(config["min_calibration_gain"]) or config["min_calibration_gain"] < 0:\n        raise ValueError("min_calibration_gain must be finite and nonnegative")\n    if type(config["min_calibration_improved_folds"]) is not int or config["min_calibration_improved_folds"] < 1:\n        raise ValueError("min_calibration_improved_folds must be a positive integer")\n    neural._settings(config)\n    if pretrained_dir is None:\n        raise ValueError("공개 사전학습 모델을 먼저 준비하고 pretrained_dir를 지정하세요.")\n    train_label_dir, base_dir, output_dir, pretrained_dir = (Path(x).expanduser().resolve() for x in\n                                             (train_label_dir, base_dir, output_dir, pretrained_dir))\n    if train_label_dir.name != "TL_서울_구급" or any("validation" in part.lower() for part in train_label_dir.parts):\n        raise ValueError("학습 입력은 서울 Training/TL_서울_구급 폴더여야 합니다.")\n    for protected in (base_dir, pretrained_dir, train_label_dir):\n        if output_dir == protected or output_dir.is_relative_to(protected) or protected.is_relative_to(output_dir):\n            raise ValueError("ver7 출력은 기존 ver4/사전학습 모델/Training과 별도 폴더여야 합니다.")\n    if config["require_cuda"] and (config["device"] != "cuda" or not torch.cuda.is_available()):\n        raise RuntimeError("ver7 학습에는 GPU 런타임을 선택하세요.")\n    pretrained_metadata = pretrained.validate_pretrained(pretrained_dir)\n    if pretrained_metadata["model_id"] != config["model_name"]:\n        raise ValueError("준비한 공개 encoder와 config.model_name이 다릅니다.")\n    (base_manifest, paths, texts, targets, fit, stop, cal, groups,\n     base_probs, sparse_probs) = _read_base(train_label_dir, base_dir, int(config["init_fold"]))\n    signature, provenance = _run_signature(config, base_manifest, base_dir, pretrained_metadata)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    run_path = output_dir / "run.json"\n    if run_path.exists() and json.loads(run_path.read_text(encoding="utf-8")).get("signature") != signature:\n        raise ValueError("ver7 출력 폴더의 데이터/설정/초기 가중치/코드가 다릅니다. 새 OUTPUT_DIR를 지정하세요.")\n    if not run_path.exists() and any(output_dir.glob("*.ckpt")):\n        raise ValueError("run.json 없는 기존 checkpoint가 있습니다. 새 OUTPUT_DIR를 지정하세요.")\n    write_json(run_path, {"signature": signature, "config": config, "provenance": provenance})\n    manifest_path = output_dir / "mission3_ver7_model.ckpt"\n    report_path = output_dir / "mission3_ver7_metrics.json"\n    if manifest_path.is_file() and report_path.is_file():\n        saved = load_manifest(manifest_path, base_dir=None if config["bundle_base"] else base_dir)\n        if saved["signature"] != signature:\n            raise ValueError("Final ver7 signature mismatch")\n        print("완료된 ver7 재사용: 추가 학습·보정 탐색을 건너뜁니다.", flush=True)\n        return manifest_path, json.loads(report_path.read_text(encoding="utf-8"))\n\n    write_json(output_dir / "splits.json", {\n        "fit": [paths[i].name for i in fit], "stop": [paths[i].name for i in stop],\n        "calibration": [paths[i].name for i in cal], "init_fold": config["init_fold"],\n        "source": "Exact stored ver4 fold for a comparable independent encoder experiment",\n    })\n    print(f"새 encoder 전체 학습 | fit={len(fit):,}, stop={len(stop):,}, calibration={len(cal):,}", flush=True)\n    print(f"{config[\'model_name\']} | 최대 {config[\'max_epochs\']} epoch | 발화 경계 SEP 보존", flush=True)\n    tokenizer = AutoTokenizer.from_pretrained(pretrained_dir, use_fast=True, local_files_only=True)\n    tokenizer.save_pretrained(output_dir / "tokenizer")\n    rendered = data.candidate_texts(texts, tokenizer)\n    documents = neural.tokenize_dialogues(rendered, tokenizer, config["max_length"], config["stride"])\n    candidate_path, done_path = output_dir / "candidate_best.ckpt", output_dir / "candidate_done.json"\n    candidate_started = time.monotonic()\n    if candidate_path.is_file() and done_path.is_file():\n        record = json.loads(done_path.read_text(encoding="utf-8"))\n        if record.get("signature") != signature or record.get("weights_sha256") != _sha256(candidate_path):\n            raise ValueError("Candidate completion signature/weights mismatch")\n        history = record["training"]\n        print("완료된 새 모델 재사용", flush=True)\n    else:\n        history = neural.train_model(\n            [documents[i] for i in fit], targets[fit], [documents[i] for i in stop], targets[stop],\n            tokenizer, config, candidate_path, seed=int(config["seed"]) + 5003,\n            initial_model_dir=pretrained_dir, run_signature=signature, stop_groups=groups[stop],\n        )\n        write_json(done_path, {"signature": signature, "training": history,\n                              "weights_sha256": _sha256(candidate_path)})\n    candidate_minutes = (time.monotonic() - candidate_started) / 60\n    _release()\n\n    # Only the corresponding old fold excluded this stop set from gradient\n    # training. The other old folds must NEVER form an ensemble stop benchmark.\n    baseline_started = time.monotonic()\n    fold_name = base_manifest["folds"][int(config["init_fold"]) - 1]\n    base_tokenizer = AutoTokenizer.from_pretrained(base_dir / base_manifest["tokenizer"],\n                                                  use_fast=True, local_files_only=True)\n    base_config = {**base_manifest["config"], "device": config["device"],\n                   "inference_batch_size": config["inference_batch_size"]}\n    base_stop_documents = base_neural.tokenize_dialogues([texts[i] for i in stop], base_tokenizer,\n                                 base_config["max_length"], base_config["stride"])\n    baseline_stop_probs = _cached_predictions(\n        output_dir / "baseline_single_fold_stop.npz", signature, [paths[i].name for i in stop],\n        provenance["base_artifact_sha256"][fold_name],\n        lambda: _predict_base_checkpoint(base_dir / fold_name, base_stop_documents, base_tokenizer, base_config),\n    )\n    baseline_stop = StopSelector(targets[stop], groups[stop], seed=int(config["seed"]) + 5003,\n                                n_splits=config["stop_crossfit_splits"]).score(baseline_stop_probs)\n    baseline_stop.update(fold=config["init_fold"],\n                         scope="Single original fold on its own stop set; repeated epoch-selection estimate")\n    baseline_minutes = (time.monotonic() - baseline_started) / 60\n    del base_stop_documents, base_tokenizer\n    _release()\n    calibration_started = time.monotonic()\n    candidate_digest = _sha256(candidate_path)\n    candidate_probs = _cached_predictions(\n        output_dir / "candidate_calibration.npz", signature, [paths[i].name for i in cal], candidate_digest,\n        lambda: neural.predict_checkpoint(candidate_path, [documents[i] for i in cal], tokenizer, config),\n    )\n    _release()\n    cal_settings = {"seed": int(base_manifest["config"]["seed"]) + 2,\n                    "n_splits": int(base_manifest["config"]["calibration_folds"])}\n    calibrator, report = calibration.calibrate_crossfit(\n        targets[cal], base_probs, sparse_probs, candidate_probs, groups[cal], **cal_settings,\n        min_gain=float(config["min_calibration_gain"]),\n        min_improved_folds=config["min_calibration_improved_folds"],\n    )\n    standalone_calibrator, standalone_report = calibration.fit_candidate_only_crossfit(\n        targets[cal], candidate_probs, groups[cal], **cal_settings,\n    )\n    report.update(standalone_report)\n    report.update({\n        "signature": signature, "base_signature": base_manifest["signature"],\n        "training_samples": len(paths), "candidate_fit_samples": len(fit), "candidate_stop_samples": len(stop),\n        "candidate_training": history, "calibrator": calibrator, "config": config,\n        "candidate_only_calibrator": standalone_calibrator, "public_pretrained": provenance["public_pretrained"],\n        "baseline_single_fold_stop": baseline_stop,\n        "candidate_minus_single_fold_stop_macro_f1": history["best_stop_macro_f1"] - baseline_stop["stop_macro_f1"],\n        "versions": _versions(), "validation_used_for_training_or_selection": False,\n        "new_models_trained": 1, "base_models_retrained": 0,\n        "timing_minutes_this_invocation": {\n            "candidate_training_or_reuse": candidate_minutes, "baseline_stop_or_reuse": baseline_minutes,\n            "calibration_prediction_and_selection": (time.monotonic() - calibration_started) / 60,\n        },\n    })\n    if config["bundle_base"]:\n        _export_base(base_dir, output_dir, base_manifest, provenance["base_artifact_sha256"])\n    else:\n        print("기존 ZIP의 ver4를 복사 없이 재사용합니다.", flush=True)\n    _export_code(output_dir)\n    manifest = {\n        "format": "mission3_ver7", "version": 1, "signature": signature,\n        "target_symptoms": list(TARGET_SYMPTOMS), "portable": config["bundle_base"],\n        "base_manifest": ("base/mission3_ver4_model.ckpt" if config["bundle_base"] else\n                          str(base_dir / "mission3_ver4_model.ckpt")),\n        "base_signature": base_manifest["signature"], "candidate": "candidate_best.ckpt",\n        "candidate_sha256": candidate_digest, "candidate_tokenizer": "tokenizer",\n        "candidate_tokenizer_sha256": _tokenizer_hashes(output_dir / "tokenizer"),\n        "config": config, "calibrator": calibrator, "versions": _versions(),\n        "candidate_only_calibrator": standalone_calibrator, "selected_name": report["selected_name"],\n    }\n    report["timing_minutes_this_invocation"]["total_pipeline"] = (time.monotonic() - started) / 60\n    write_json(report_path, report)\n    _atomic_torch_save(manifest_path, manifest)\n    print(f"Training 보정 교차평가: ver4={report[\'baseline_crossfit_macro_f1\']:.6f}, "\n          f"새 모델 단독={report[\'candidate_standalone_crossfit_macro_f1\']:.6f}, "\n          f"혼합 후보={report[\'candidate_crossfit_macro_f1\']:.6f}, 선택={report[\'selected_name\']}", flush=True)\n    print("Training 내부 선택용 점수입니다. 공식 Validation은 선택 완료 후 별도 평가합니다.", flush=True)\n    return manifest_path, report\n\n\ndef load_manifest(checkpoint_path, base_dir=None):\n    """Validate a portable bundle or an explicit external-base experiment.\n\n    ``base_dir`` relocates an external ver4 dependency without rewriting the\n    saved manifest. Portable manifests always retain their internal base.\n    """\n    checkpoint_path = Path(checkpoint_path).resolve()\n    manifest = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    if manifest.get("format") != "mission3_ver7" or manifest.get("version") != 1:\n        raise ValueError("mission3_ver7_model.ckpt를 지정하세요.")\n    if manifest.get("target_symptoms") != list(TARGET_SYMPTOMS):\n        raise ValueError("ver7 target label order mismatch")\n    root = checkpoint_path.parent\n    candidate = (root / manifest["candidate"]).resolve()\n    if not candidate.is_relative_to(root) or not candidate.is_file():\n        raise FileNotFoundError(f"ver7 결과 폴더 전체를 함께 보관하세요: {manifest[\'candidate\']}")\n    if _sha256(candidate) != manifest.get("candidate_sha256"):\n        raise ValueError("ver7 candidate weights hash mismatch")\n    tokenizer_dir = (root / manifest["candidate_tokenizer"]).resolve()\n    if (not tokenizer_dir.is_relative_to(root) or not tokenizer_dir.is_dir()\n            or _tokenizer_hashes(tokenizer_dir) != manifest.get("candidate_tokenizer_sha256")):\n        raise ValueError("ver7 candidate tokenizer hash/path mismatch")\n    portable = manifest.get("portable", True)\n    if not isinstance(portable, bool):\n        raise ValueError("Invalid ver7 portable flag")\n    if portable:\n        if base_dir is not None:\n            raise ValueError("--base_dir is only supported for external-base ver7 experiments")\n        base_path = (root / manifest["base_manifest"]).resolve()\n        if not base_path.is_relative_to(root) or not base_path.is_file():\n            raise FileNotFoundError(f"ver7 결과 폴더 전체를 함께 보관하세요: {manifest[\'base_manifest\']}")\n    else:\n        base_path = Path(manifest["base_manifest"])\n        if not base_path.is_absolute():\n            raise ValueError("External ver4 manifest path must be absolute")\n        if base_dir is not None:\n            base_path = Path(base_dir).expanduser().resolve() / "mission3_ver4_model.ckpt"\n        if not base_path.is_file():\n            raise FileNotFoundError("기존 ver4 묶음을 다시 준비하고 --base_dir에 폴더를 지정하세요: "\n                                    f"{base_path}")\n        manifest["base_manifest"] = str(base_path.resolve())\n    base = base_pipeline.load_manifest(base_path)\n    if base.get("signature") != manifest.get("base_signature"):\n        raise ValueError("ver7 references a different ver4 base")\n    return manifest\n\n\ndef _predict_base_checkpoint(path, documents, tokenizer, config):\n    # Keep archived ver4 source bytes intact for its data/code fingerprint.\n    # Its old support check can count emulated BF16 on T4; use native precision\n    # during this synchronous inference call and restore the original helper.\n    original = base_neural._amp\n    base_neural._amp = neural._amp\n    try:\n        return base_pipeline.predict_checkpoint(path, documents, tokenizer, config)\n    finally:\n        base_neural._amp = original\n\n\ndef _cache_execution(device):\n    # Keep the original cache schema so the unchanged ver4 branch is reusable.\n    execution = {"versions": {**_versions(), **{name: importlib.metadata.version(name)\n                 for name in ("scipy", "joblib")}}, "cuda_version": torch.version.cuda}\n    if device == "cuda":\n        execution.update(gpu=torch.cuda.get_device_name(), capability=list(torch.cuda.get_device_capability()),\n                         bf16=neural.native_bf16_supported())\n    return execution\n\n\ndef predict_pipeline(checkpoint_path, label_dir, output, device=None, batch_size=None, cache_dir=None,\n                     base_dir=None, candidate_output=None):\n    checkpoint_path = Path(checkpoint_path).resolve()\n    if candidate_output is not None and Path(candidate_output).resolve() == Path(output).resolve():\n        raise ValueError("candidate_output must differ from final output")\n    manifest = load_manifest(checkpoint_path, base_dir=base_dir)\n    root = checkpoint_path.parent\n    base_path = root / manifest["base_manifest"]\n    base_manifest = base_pipeline.load_manifest(base_path)\n    _check_versions(base_manifest)\n    base_root = base_path.parent\n    config = dict(manifest["config"])\n    config["device"] = device or ("cuda" if torch.cuda.is_available() else "cpu")\n    if batch_size is not None:\n        if int(batch_size) < 1:\n            raise ValueError("batch_size must be positive")\n        config["inference_batch_size"] = int(batch_size)\n    paths, texts, _ = load_dataset(Path(label_dir), include_targets=False)\n    base_config = {**base_manifest["config"], "device": config["device"],\n                   "inference_batch_size": config["inference_batch_size"]}\n    base_tokenizer = AutoTokenizer.from_pretrained(base_root / base_manifest["tokenizer"],\n                                                 use_fast=True, local_files_only=True)\n    execution = _cache_execution(config["device"])\n    base_cache = (PredictionCache(cache_dir, paths, texts, base_root / base_manifest["tokenizer"],\n                                 TARGET_SYMPTOMS, execution) if cache_dir is not None else None)\n    source = Path(__file__).parent\n\n    def predict(cache, kind, path, settings, sources, compute):\n        if cache is None:\n            return compute()\n        return cache.get(kind, path, settings, [source / name for name in sources], compute)\n\n    alpha = np.asarray(manifest["calibrator"]["neural_weights"])\n    beta = np.asarray(manifest["calibrator"]["candidate_weights"])\n    base_probs = np.zeros((len(paths), len(TARGET_SYMPTOMS)), dtype=np.float64)\n    sparse_probs = np.zeros_like(base_probs)\n    candidate_probs = None\n    if np.any((1 - beta) * alpha > 0):\n        base_documents = base_neural.tokenize_dialogues(texts, base_tokenizer,\n                                      base_config["max_length"], base_config["stride"])\n        for name in base_manifest["folds"]:\n            print("Predict base:", name, flush=True)\n            base_probs += predict(base_cache, "ver4-neural", base_root / name, base_config,\n                                  ["ver4_runtime/neural.py"],\n                                  lambda: _predict_base_checkpoint(base_root / name, base_documents, base_tokenizer, base_config))\n            _release()\n        base_probs /= len(base_manifest["folds"])\n        del base_documents\n    if np.any((1 - beta) * (1 - alpha) > 0):\n        sparse_probs = predict(base_cache, "ver4-sparse", base_root / base_manifest["sparse"], base_config,\n                               ["ver4_runtime/pipeline.py"],\n                               lambda: base_pipeline.predict_sparse(joblib.load(base_root / base_manifest["sparse"]), texts))\n    if np.any(beta > 0) or candidate_output is not None:\n        # New tokenizer + SEP rendering must never replace the original baseline inputs.\n        tokenizer_dir = root / manifest["candidate_tokenizer"]\n        candidate_tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir, use_fast=True, local_files_only=True)\n        rendered = data.candidate_texts(texts, candidate_tokenizer)\n        candidate_documents = neural.tokenize_dialogues(rendered, candidate_tokenizer,\n                                                        config["max_length"], config["stride"])\n        candidate_cache = (PredictionCache(cache_dir, paths, rendered, tokenizer_dir, TARGET_SYMPTOMS, execution)\n                           if cache_dir is not None else None)\n        print("Predict: candidate_best.ckpt", flush=True)\n        candidate_probs = predict(candidate_cache, "ver7-neural", root / manifest["candidate"], config,\n                                  ["neural.py", "data.py", "ver4_runtime/neural.py"],\n                                  lambda: neural.predict_checkpoint(root / manifest["candidate"], candidate_documents,\n                                                                    candidate_tokenizer, config))\n        _release()\n    probabilities = calibration.apply_calibrator(base_probs, sparse_probs, candidate_probs, manifest["calibrator"])\n    predictions = write_predictions(paths, probabilities, manifest["calibrator"]["thresholds"], Path(output))\n    if candidate_output is not None:\n        write_predictions(paths, candidate_probs, manifest["candidate_only_calibrator"]["thresholds"], Path(candidate_output))\n        print(f"새 모델 단독 예측 저장: {candidate_output} (Training에서 고정한 임계값)", flush=True)\n    print(f"{len(paths):,} predictions saved: {output}", flush=True)\n    return paths, probabilities, predictions\n\n\nevaluate_csv = base_pipeline.evaluate_csv\n', 'train.py': '"""Train the complementary model and select calibration using Training only."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport torch\n\nif __package__:\n    from .pipeline import train_pipeline\nelse:\n    from pipeline import train_pipeline\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--train_label_dir", required=True)\n    parser.add_argument("--base_dir", required=True)\n    parser.add_argument("--pretrained_dir", required=True, help="Prepared public KoELECTRA encoder/tokenizer directory.")\n    parser.add_argument("--output_dir", required=True)\n    parser.add_argument("--config_json")\n    parser.add_argument("--auto_batch", action="store_true")\n    args = parser.parse_args()\n    config = json.loads(Path(args.config_json).read_text(encoding="utf-8")) if args.config_json else {}\n    if args.auto_batch:\n        run_path = Path(args.output_dir) / "run.json"\n        if run_path.is_file():\n            # Keep the original batch schedule. Incomplete neural training also\n            # verifies the GPU/precision identity before reusing epoch states.\n            saved = json.loads(run_path.read_text(encoding="utf-8"))["config"]\n            for key in ("batch_size", "window_batch_size", "gradient_accumulation_steps"):\n                config.setdefault(key, saved[key])\n            print("이전 실행의 batch 설정을 유지하여 재개합니다.", flush=True)\n        elif torch.cuda.is_available():\n            memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)\n            batch = 16 if memory_gb >= 35 else 8 if memory_gb >= 22 else 4\n            config.setdefault("batch_size", batch)\n            config.setdefault("window_batch_size", batch)\n            config.setdefault("gradient_accumulation_steps", 32 // config["batch_size"])\n    print("ver7 추가 학습 설정:", json.dumps(config, ensure_ascii=False), flush=True)\n    checkpoint, report = train_pipeline(args.train_label_dir, args.base_dir, args.output_dir, config,\n                                        pretrained_dir=args.pretrained_dir)\n    print("Checkpoint:", checkpoint, flush=True)\n    print("선택:", report["selected_name"], flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'inference.py': '"""One-command Mission 3 inference; only transcript text is a model input."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nif __package__:\n    from .pipeline import evaluate_csv, predict_pipeline\n    from .ver4_runtime.data import write_json\nelse:\n    from pipeline import evaluate_csv, predict_pipeline\n    from ver4_runtime.data import write_json\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--audio_dir", default="", help="Accepted for the competition interface; text task does not read audio.")\n    parser.add_argument("--label_dir", required=True)\n    parser.add_argument("--ckpt_path", required=True)\n    parser.add_argument("--output", required=True)\n    parser.add_argument("--device", choices=("cuda", "cpu"))\n    parser.add_argument("--batch_size", type=int)\n    parser.add_argument("--cache_dir", help="Optional persistent raw prediction cache shared across experiments.")\n    parser.add_argument("--base_dir", help="Relocated ver4 folder for an external-base ver7 experiment.")\n    parser.add_argument("--candidate_output", help="Also save the new model alone using Training-calibrated thresholds.")\n    parser.add_argument("--evaluate", action="store_true", help="Evaluate only after CSV is saved and all targets exist.")\n    args = parser.parse_args()\n    paths, _, _ = predict_pipeline(args.ckpt_path, args.label_dir, args.output, args.device, args.batch_size,\n                                   cache_dir=args.cache_dir, base_dir=args.base_dir,\n                                   candidate_output=args.candidate_output)\n    if args.evaluate:\n        target_count = sum("symptom" in json.loads(path.read_text(encoding="utf-8-sig")) for path in paths)\n        if target_count == len(paths):\n            report = evaluate_csv(args.label_dir, args.output)\n            write_json(Path(args.output).with_name("validation_metrics.json"), report)\n            print(f"Final evaluation Macro F1: {report[\'macro_f1\']:.6f}", flush=True)\n            for name, values in report["per_symptom"].items():\n                print(f"  {name}: {values[\'f1\']:.6f}", flush=True)\n            if args.candidate_output:\n                candidate_report = evaluate_csv(args.label_dir, args.candidate_output)\n                write_json(Path(args.candidate_output).with_name("candidate_validation_metrics.json"), candidate_report)\n                print(f"Candidate-only evaluation Macro F1: {candidate_report[\'macro_f1\']:.6f}", flush=True)\n        else:\n            print(f"정답 포함 JSON {target_count}/{len(paths)}: 예측 완료, 평가는 생략합니다.", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'requirements.txt': 'torch>=2.6,<3\ntransformers==4.57.1\nnumpy==2.1.3\nscikit-learn==1.6.1\nscipy>=1.11,<2\niterative-stratification==0.1.9\njoblib>=1.3,<2\n', 'README.md': '# Mission 3 ver7\n\n공개 사전학습 **KoELECTRA-Base-v3에서 시작해 전체 encoder와 새 9-label attention 분류기**를 함께 학습한다.\n기존 ver4의 5개 모델은 비교와 혼합에 재사용한다. 실행 파일은 `src/mission3/mission3_ver7.ipynb` 하나이고,\n기존 **`mission3_ver5_inputs.zip`을 그대로 사용**한다. 새 입력 ZIP이나 ver5·ver6 결과는 필요 없다.\n\n## Colab에서 모두 실행\n\n1. Drive 공유 문서함의 `mission3_ver5_inputs.zip`에서 **정리 → 바로가기 추가 → 내 드라이브**를 선택한다.\n2. 내 드라이브 바로 아래에 같은 이름의 바로가기를 둔다.\n3. `mission3_ver7.ipynb`를 Colab에서 열고 GPU 런타임을 선택한 뒤 **모두 실행**한다.\n4. 바로가기를 만든 계정으로 Drive를 연결한다.\n\n바로가기를 하위 폴더에 두었다면 첫 셀의 `INPUT_ZIP`만 바꾼다. 별도 DATA 폴더 연결이나 WAV는 필요 없다.\nDrive 연결 후 바로가기를 추가하여 파일이 보이지 않으면 경로 셀의 `force_remount=True`로 한 번 재연결한다.\n필요한 소스는 노트북에 모두 포함되어 있다.\n\n| 용도 | 기본 경로 |\n|---|---|\n| 입력 ZIP 바로가기 | `/content/drive/MyDrive/mission3_ver5_inputs.zip` |\n| 기존 모델 복원 | `/content/mission3_ver5_inputs/DCC_AIdeas/results/mission3/ver4` |\n| Training·Validation JSON | `/content/mission3_ver5_inputs/DCC_AIdeas/DATA/대학부 데이터/` |\n| 새 공개 encoder·tokenizer | `/content/mission3_ver7_pretrained/koelectra_base_v3` |\n| **새 결과** | **`/content/drive/MyDrive/DCC_AIdeas/results/mission3/ver7`** |\n| 공유 원시 예측 캐시 | `/content/drive/MyDrive/DCC_AIdeas/results/mission3/prediction_cache` |\n\n`ver4` 입력 폴더의 이름을 바꾸지 않는다. 원래 ZIP은 읽기 전용으로 사용하며 manifest·CRC·SHA256을 검사한다.\n같은 런타임에서 검증된 압축 해제 파일은 재사용한다. 새 Colab 세션이면 로컬 복원을 다시 한다.\nver7은 전체 encoder를 학습하므로 frozen feature cache를 만들지 않는다.\n\n## 공개 사전학습 모델 준비\n\n[`monologg/koelectra-base-v3-discriminator`](https://huggingface.co/monologg/koelectra-base-v3-discriminator)의\n공식 [커밋 `68b30cd259f34a4b5aa8786392612ba2a2617fcc`](https://huggingface.co/monologg/koelectra-base-v3-discriminator/commit/68b30cd259f34a4b5aa8786392612ba2a2617fcc)을 고정한다.\n처음에는 encoder 가중치 약 452 MB와 tokenizer를 자동 다운로드한다. 공개 모델 준비 단계에만 Hugging Face\n네트워크를 허용하고, 대회 JSON이나 전사 text는 전달하지 않는다. 이후 학습·추론은 오프라인이다.\n\n`AutoModel`로 encoder를 읽어 `safetensors`로 저장하고, 로컬 재로딩과 파일별 SHA256 검사를 통과한 뒤 완료 manifest를 만든다.\n완료 폴더의 모든 파일을 검증하고 재사용하므로 같은 런타임에서 불필요하게 다시 다운로드하지 않는다.\n모델·tokenizer는 새 후보용이고, 기존 5개 모델은 ZIP에 들어 있는 원래 tokenizer를 계속 사용한다.\n\n## 실험과 데이터 분할\n\n모든 발화 text를 원래 순서대로 동일한 `[SEP]`로 구분한다. 정답과 speaker에 따라 입력을 바꾸지 않는다.\n긴 대화를 512-token window와 64-token overlap으로 처리하고 증상별 token attention에서 9개 확률을 직접 계산한다.\n전체 encoder와 분류기를 BCE로 학습한다. 출처와 적용 범위는 [RESEARCH.md](RESEARCH.md)에 기록한다.\n\n| 구간 | 역할 |\n|---|---|\n| 원래 fold 1 fit 21,032개 | 가중치 학습 |\n| 원래 stop 5,288개 | 그룹을 분리한 3-fold threshold 교차평가로 epoch 선택 |\n| 같은 stop의 원래 ver4 fold 1 | 기존 단일 모델과의 비교 진단 |\n| 별도 Training calibration 2,880개 | 5-fold 교차평가로 기존 앙상블·새 후보 혼합과 임계값 선택 |\n| 공식 Validation 3,640개 전체 | 선택 확정 후 최종 구성·후보 단독 평가 |\n\n최종 혼합 후보는 기존 calibration 교차평가 Macro F1보다 **0.002 이상** 좋고,\n**5개 fold 중 최소 3개**에서 엄격히 개선되어야 채택한다. 미달·동점이면 기존 ver4 보정을 유지한다.\n후보 단독 calibration 결과도 기록한다. Validation은 학습·보정·선택에 사용하지 않는다.\n\n이 실험은 새 encoder 하나의 고정 fit/stop 실험이다. 5개 새 encoder를 각각 학습한 모델 OOF가 아니다.\n같은 stop에서 여러 epoch를 비교한 선택 편향과 calibration 반복 연구의 선택 편향이 남는다.\n최소 개선 기준은 Validation·Test 성능의 상승을 보장하지 않는다.\n\n## 기본 설정·재개\n\n| 설정 | 기본값 |\n|---|---|\n| 최대 epoch / patience | 3 / 2 |\n| encoder / classifier learning rate | `2e-5` / `1e-4` |\n| 유효 batch | 32 |\n| GPU 자동 microbatch | T4 4 / L4 8 / A100 16 |\n| Validation 최종 추론 batch | 8 |\n| `RUN_VALIDATION`, `RUN_CANDIDATE_DIAGNOSTIC` | `True`, `True` |\n| `bundle_base` | `False` |\n\n`--auto_batch`가 GPU 메모리에 맞춰 microbatch와 window batch를 정하고 gradient accumulation으로 유효 batch를 유지한다.\n**미완료 학습 재개는 같은 GPU 종류·AMP 정밀도·관련 패키지 버전과 같은 코드·데이터·설정이 필요하다.**\n완료된 최종 결과는 다른 GPU에서도 재사용할 수 있으며, 조건이 다른 원시 예측 캐시는 다시 계산한다.\n새 설정 실험은 다른 `OUTPUT_DIR`을 사용한다. 실제 GPU 학습 시간과 CU는 아직 실측하지 않았다.\n\n## 결과 읽기\n\n- `mission3_ver7_model.ckpt`: 확정한 모델 구성·혼합·임계값 manifest.\n- `candidate_best.ckpt`: 선택한 epoch의 새 encoder와 attention 분류기 가중치.\n- `mission3_ver7_metrics.json`: 후보 학습 이력, 원래 fold 1 단독 비교, 내부 교차평가, 채택/fallback 이유.\n- `predictions/<실행시각>/mission3.csv`, `validation_metrics.json`: 최종 출력과 전체·증상별 precision/recall/F1/support.\n- 같은 폴더의 `candidate_only.csv`, `candidate_validation_metrics.json`: **새 후보 단독 진단**.\n- `notebook_runtime_<실행시각>.json`: 입력 준비, 공개 모델 준비, 학습·보정, 추론·평가 시간. 패키지 설치와 CU는 제외.\n- `submission/`: 실행 가능한 추론 소스·패키지 목록·문서.\n\n후보 단독 진단의 임계값은 Training calibration에서 고정한다. 진단 점수로 모델·혼합·임계값을 다시 고르지 않는다.\n최종 구성에 후보가 제외되어도 진단을 위해 후보를 추가 추론하며, 관련 원시 예측 캐시가 있으면 재사용한다.\n이전 전체 Validation Macro F1은 ver4 **0.6566489480388843**, ver5 **0.6567177583831921**,\nver6 **0.6566697183939796**이다. 노트북 마지막에 각각과 차이를 표시한다. 현재 GPU·batch 차이로 작은 수치 차이가 생길 수 있다.\n\n기본 `bundle_base=False`는 기존 5개 모델을 Drive에 중복 복사하지 않는다. **원래 입력 ZIP과 ver7 결과를 함께 보관한다.**\n새 후보는 결과 폴더에 포함되므로 최종 추론에는 공개 encoder 다운로드가 필요하지 않다.\n\n```bash\npython submission/inference.py --audio_dir "" --label_dir "JSON 폴더" \\\n  --ckpt_path mission3_ver7_model.ckpt --base_dir "ZIP에서 복원한 ver4 폴더" --output mission3.csv\n```\n\n`--audio_dir`는 대회 인터페이스 호환용이다. `--evaluate`는 CSV 확정 후 평가만 수행하고,\n`--candidate_output candidate_only.csv`는 후보 단독 진단을 추가한다. `--cache_dir`로 공용 원시 예측 캐시를 지정할 수 있다.\n캐시에는 정답·보정값을 저장하지 않으며 text·가중치·tokenizer·추론 코드·관련 실행 조건이 같은 경우에만 재사용한다.\n\n노트북 재생성: `python src/mission3/mission3_ver7/build_notebook.py`.\n작은 CPU 테스트는 구현·재개·저장·캐시 동작 검증용이며 실제 전체 데이터 F1 검증을 대신하지 않는다.\n', 'RESEARCH.md': '# Mission 3 ver7: 설계 근거와 검증 범위\n\n검토일: 2026-09-15. ver7은 새로운 공개 한국어 encoder의 전체 미세조정이 주는 효과를 확인한다.\nver4의 학습된 encoder 위에 작은 분류기를 바꾸던 이전 실험에서 출발점을 바꾼다.\n실제 전체 Validation F1·Colab 시간·CU는 아직 측정하지 않았다.\n\n## 확인한 1차 출처\n\n| 출처 | 확인 사항 | ver7 적용 |\n|---|---|---|\n| [KoELECTRA 저자 저장소](https://github.com/monologg/KoELECTRA) | Base discriminator 12층·hidden 768, v3의 추가 사전학습 자료에 구어·메신저 포함 | 한국어 대화 표현에 대한 실험 가설로 공개 v3 encoder 선택 |\n| [공식 모델 카드](https://huggingface.co/monologg/koelectra-base-v3-discriminator) · [고정 커밋](https://huggingface.co/monologg/koelectra-base-v3-discriminator/commit/68b30cd259f34a4b5aa8786392612ba2a2617fcc) | `ElectraModel` encoder 사용법, Apache-2.0 표시, tokenizer | 해당 encoder·tokenizer만 다운로드하고 버전 고정 |\n| Huang et al., [PLM-ICD, ClinicalNLP 2022](https://aclanthology.org/2022.clinicalnlp-1.2.pdf), §4.2–4.3 | 긴 문서 segment의 token 표현을 합치고 label별 attention으로 분류 | 긴 전사 window와 9개 증상별 attention, encoder와 분류기 공동 학습 |\n\nKoELECTRA v3에 구어·메신저가 포함되었다는 사실이 Mission 3 향상을 뜻하지는 않는다.\n새 encoder의 효과는 같은 원래 fold 1 fit/stop에서 기존 단일 모델과 비교하고,\nTraining calibration에서 후보 단독·기존 앙상블 혼합을 따로 비교한다.\n\n## 구현 범위\n\n```text\n모든 발화 text → 동일한 [SEP]로 순서대로 연결 → overlapping windows\n  → 공개 KoELECTRA encoder 전체 미세조정\n  → window들의 token 표현 → 9-label attention → 9개 logits → sigmoid\n```\n\n512-token window와 64-token overlap을 사용한다. attention은 padding과 special token을 제외한 표현을 선택한다.\n겹친 token은 서로 다른 window 문맥에서 계산된 표현으로 포함한다. BCE로 encoder와 분류기를 함께 학습한다.\nPLM-ICD의 영어 임상 사전학습·대규모 ICD label 체계·의료 corpus와 원문 전체 실험을 재현하는 구현은 아니다.\n증상별 attention 가중치를 원인 설명으로 해석하지 않는다.\n\n학습 데이터는 제공된 서울 Training이다. 모든 전사 text를 동일한 규칙으로 처리하며 speaker 필드와\n증상 정답에 따른 입력 변경은 없다. 별도 외부 dataset, 의료 사전, LLM 생성 데이터, 재라벨링을 추가하지 않는다.\n공개 사전학습 encoder를 가져오는 것은 새로운 지도학습 데이터 추가와 구분한다.\n\n## 비교와 선택\n\n가중치 학습은 기존 fit 21,032개, epoch 비교는 기존 stop 5,288개,\n최종 혼합·임계값 선택은 별도 Training calibration 2,880개를 사용한다.\nstop의 threshold는 대화 그룹을 나눈 3-fold에서 평가 fold 바깥으로 맞춘다.\n같은 stop의 원래 ver4 fold 1 단독 점수는 기존 앙상블 점수와 구분하여 기록한다.\n\ncalibration에서는 5-fold 교차평가로 기존 앙상블과 새 혼합을 비교한다.\n최종 개선 폭 0.002 이상과 5개 중 3개 fold의 엄격한 개선을 함께 요구한다.\n이 기준은 연구용 보수적 채택 조건이며 통계적 유의성 검정이나 일반화 보장은 아니다.\n하나의 고정 fold에서 epoch를 고르므로 완전한 모델 OOF가 아니고 내부 선택 편향이 남는다.\n\n공식 Validation 3,640개는 최종 선택 이후 평가한다. 최종 구성과 후보 단독의 두 CSV를 확정하고 점수를 기록한다.\n후보 단독 임계값 역시 Training calibration에서 먼저 고정되며, Validation 진단 결과는 선택에 전달하지 않는다.\n선택 로직·모델 코드·입력/실행 조건이 바뀌면 새 출력 폴더에서 별도 실험으로 다룬다.\n\n## 재현성과 비용\n\n공개 encoder는 commit SHA를 고정하고 준비 파일의 SHA256을 기록한다. 다운로드 직후\nsafetensors·config·tokenizer를 로컬로 다시 열어 본 뒤 원자적으로 완료 폴더를 게시한다.\n이후 학습·추론은 네트워크를 차단한 별도 프로세스에서 수행한다.\n\n새 encoder는 최대 3 epoch 전체 미세조정하므로 이전 분류기 중심 실험과 비용이 다르다.\n원래 5개 모델은 재학습하지 않지만, 캐시가 없으면 기존 모델들의 추론 비용이 든다.\n기존 fold 1 stop 비교와 후보 단독 Validation 진단의 추가 순전파 비용도 포함된다.\n노트북은 준비·다운로드·학습·추론의 벽시계 시간을 기록하며 CU는 Colab에서 별도로 측정한다.\n작은 CPU 테스트는 파일 무결성·학습 경로·재개·캐시·평가 분리를 확인한다. 대규모 GPU 결과를 대신하지 않는다.\n', 'ver4_runtime/README.md': '# Mission 3 ver4 실행 안내\n\n실행 파일은 [src/mission3_ver4.ipynb](../mission3_ver4.ipynb)입니다. 이 노트북 하나만 Colab에 올리면 필요한 모델·학습·추론 코드를 생성합니다. 이 폴더의 `.py` 파일은 같은 기능을 로컬이나 제출 환경에서 실행하기 위한 코드입니다.\n\n## 기존 결과와 개선 근거\n\n| 실행 버전 | 실제 Validation Macro F1 | 근거 |\n|---|---:|---|\n| `mission3_all_in_one_ver2` → results/mission3/ver1 | 0.600123 | `results/mission3/ver1/mission3_metrics.json` |\n| `mission3_ver3` → results/mission3/ver3 | 0.626015 | `results/mission3/ver3/mission3_ver3_metrics.json` |\n| ver4 | 아직 미측정 | 전체 GPU 학습 후 평가 필요 |\n\nver3의 Training 내부 F1은 3 epoch에서 0.644361로 최고였고, 15 epoch에는 0.601536으로 하락했습니다. 학습 손실은 계속 감소해 과적합 징후가 있습니다. ver4는 별도 stopping 집합의 Macro AP로 최적 epoch를 선택하고 조기 종료합니다. AP는 임계값 없이 예측 순위를 평가하며, 최종 대회 지표는 그대로 **9개 증상의 Macro F1**입니다.\n\nTraining 29,200개를 저장된 ver3 tokenizer로 직접 측정했을 때 384토큰을 넘는 대화는 3,292개(11.27%), 512토큰을 넘는 대화는 788개(2.70%)였습니다. 특수 토큰을 포함한 최대 길이는 873토큰입니다. ver4는 긴 대화의 뒷부분도 처리하도록 설계했습니다.\n\n**현재 작성 환경에는 GPU가 없어 ver4 전체 학습과 Validation 성능은 확인하지 못했습니다.** 작은 CPU 모델을 이용한 실행 검증은 코드 경로 확인용입니다. 기존 버전보다 성능이 높다는 판단은 전체 학습·추론이 끝난 뒤 실제 점수로 해야 합니다.\n\n## 모델과 데이터 분리\n\n- `klue/roberta-large`를 5개 fold에서 각각 학습하고 예측 확률을 평균냅니다.\n- 한 window는 최대 512토큰이며 인접 window는 64토큰이 겹칩니다. 마지막 window까지 유지하고, 증상별 attention으로 대화 전체를 모은 뒤 **통화당 한 번** BCE 손실을 계산합니다.\n- 문자 2–5그램과 단어 1–2그램 TF-IDF + 증상별 Logistic Regression을 함께 학습합니다.\n- 증상마다 신경망/TF-IDF의 혼합 비율과 판정 임계값을 Training 내부에서 결정합니다. 어떤 통화도 최소 1개 증상을 출력하도록 강제하지 않습니다.\n\n서울 Training을 약 90% development와 10% calibration으로 먼저 나눕니다. Development 내부 5-fold는 원래 Training 기준으로 각 fold마다 약 72% fit, 18% stopping입니다. 동일한 정규화 대화는 같은 그룹에 두어 분할을 넘지 않게 합니다.\n\nCalibration 10%는 신경망 학습, TF-IDF 학습, epoch 선택에 전혀 사용하지 않습니다. TF-IDF는 development 90%에만 fit합니다. 완성된 5개 신경망의 평균 확률과 TF-IDF 확률을 calibration에 적용한 뒤, 그 안에서 다시 5-fold 교차 보정하여 혼합 비율·임계값 선택의 일반화 점수를 계산합니다. 배포용 보정값은 calibration 전체로 결정합니다. **보정 후 기본 모델을 다시 학습하지 않고 같은 모델들을 그대로 추론에 사용합니다.** 공식 Validation은 이 과정에 들어가지 않습니다.\n\n주요 기본값은 `pipeline.py`의 `DEFAULT_CONFIG`에 있습니다.\n\n| 설정 | 기본값 |\n|---|---|\n| 모델 / fold | `klue/roberta-large` / 5 |\n| 최대 epoch / 조기 종료 patience | 6 / 2 |\n| 통화 batch / gradient accumulation | 2 / 16 |\n| encoder / head learning rate | `1.5e-5` / `1e-4` |\n| 문자 / 단어 최대 특징 수 | 120,000 / 40,000 |\n\n## Colab에서 실행\n\n1. [mission3_ver4.ipynb](../mission3_ver4.ipynb)를 Colab에 업로드합니다.\n2. GPU 런타임을 선택합니다. 기본 large 모델의 5-fold 학습에는 **A100 또는 L4를 권장**합니다. 사용 가능한 장치에 따라 시간과 메모리 사용량은 달라집니다.\n3. 제공 데이터의 압축을 풀고 `TL_서울_구급`, `VL_서울_구급` 폴더 이름을 유지합니다. Mission 3은 JSON만 읽습니다.\n4. 위에서부터 모든 셀을 실행하고 Google Drive를 연결합니다. 노트북은 `/content`를 먼저 탐색하고 Drive의 알려진 데이터 경로를 확인합니다. 자동 탐색에 실패하면 노트북의 데이터 경로 설정에 실제 폴더를 지정합니다.\n5. 학습·보정이 끝난 뒤 Validation 추론과 평가 셀을 실행합니다. 실행 로그가 남은 노트북도 저장합니다.\n\n기본 결과 폴더는 Colab에서 `/content/drive/MyDrive/DCC_AIdeas/results/mission3/ver4`, 로컬에서 프로젝트의 `results/mission3/ver4`입니다. 신경망 가중치 5개를 저장하므로 Drive 여유 공간도 확인합니다. 패키지와 최초 공개 pretrained 모델 다운로드에는 인터넷 연결이 필요하며, 학습 완료 후에는 저장된 구성으로 오프라인 추론할 수 있습니다.\n\n## 로컬 또는 제출 환경에서 실행\n\n다음 명령은 `DCC_AIdeas` 프로젝트 루트 기준입니다. 전체 학습 명령은 CUDA GPU를 요구합니다.\n\n```bash\npython -m pip install -r src/mission3_ver4/requirements.txt\n\npython src/mission3_ver4/train.py \\\n  --label_dir "DATA/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급" \\\n  --output_dir results/mission3/ver4\n\npython src/mission3_ver4/inference.py \\\n  --audio_dir "DATA/대학부 데이터/Validation/1.원천데이터/VS_서울_구급" \\\n  --label_dir "DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급" \\\n  --ckpt_path results/mission3/ver4/mission3_ver4_model.ckpt \\\n  --output results/mission3/ver4/mission3.csv\n\npython src/mission3_ver4/evaluate.py \\\n  --label_dir "DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급" \\\n  --prediction results/mission3/ver4/mission3.csv \\\n  --output results/mission3/ver4/validation_metrics.json\n```\n\n`--audio_dir`는 주최 측 인터페이스를 맞추기 위해 받으며 음성 파일을 읽지 않습니다. 추론은 `--device cpu`도 지원하지만 large 모델 5개를 처리하므로 GPU보다 느릴 수 있습니다. 학습 설정은 `train.py --config 설정.json`으로 변경할 수 있습니다. 설정을 바꿀 때는 새 `--output_dir`를 사용합니다.\n\n## 결과 읽기와 파일 보관\n\n| 파일 | 의미 |\n|---|---|\n| `mission3_ver4_metrics.json` | Training 내부 보정 점수, 증상별 교차 보정 F1, 최종 보정값 |\n| `mission3_ver4_history.json` | 각 fold의 epoch별 학습·stopping 기록 |\n| `validation_metrics.json` | 위 평가 명령으로 만든 공식 Validation Macro F1와 증상별 점수 |\n| `mission3.csv` | `label file name,symptom` 형식의 예측 결과 |\n| `run.json`, `splits.json` | 실행 설정·데이터/코드 식별값과 분할 파일 목록 |\n| `submission/` | 실행 당시 `.py` 파일, requirements와 실행 안내 사본 |\n\n`calibration_crossfit_macro_f1`는 각 보정 평가 샘플의 정답을 보정값 선택에서 제외하여 계산한 **Training 내부 점수**입니다. `calibration_fit_macro_f1`는 같은 calibration 데이터에서 보정값을 정하고 다시 평가한 점수라 낙관적일 수 있습니다. 둘을 Validation 점수로 보고하면 안 됩니다. 공식 Validation 평가를 모델 선택이나 임계값 재조정에 반복 사용하지 않습니다.\n\n**`mission3_ver4_model.ckpt`는 구성 목록과 보정값을 담은 manifest입니다. 이 파일 하나만 옮기면 추론할 수 없습니다.** 결과 디렉터리 전체를 함께 보관하거나 최소한 다음 구성을 유지합니다.\n\n```text\nver4/\n├─ mission3_ver4_model.ckpt\n├─ fold_1.ckpt ... fold_5.ckpt\n├─ tokenizer/\n├─ sparse.joblib\n└─ submission/\n```\n\nCSV의 symptom 값은 기존 결과와 같은 문자열 목록 표현입니다. 예를 들어 `[]`, `[\'구토\', \'복통\']`처럼 저장하며 CSV 인용 처리는 코드가 담당합니다. 주최 측 sample submission에서 더 구체적인 형식을 제공하면 해당 규격도 확인합니다.\n\n## 중단 후 다시 실행\n\n같은 데이터·설정·코드와 같은 결과 폴더로 다시 실행하면 `fold_N.ckpt`와 `fold_N_done.json`이 모두 존재하는 **완료된 fold만 재사용**합니다. 실행 도중 끊긴 fold는 처음부터 다시 학습합니다. 마지막 미니배치나 optimizer 상태에서 이어가는 기능은 아닙니다.\n\n데이터·설정·코드의 식별값이 기존 실행과 다르면 혼합된 결과를 만들지 않도록 중단합니다. 이 경우 새 결과 폴더를 지정합니다. 메모리가 부족하면 통화 batch와 window batch를 줄이는 설정을 검토하고 새 폴더에서 시작합니다.\n\n## 대회 조건과 참고 자료\n\n판단 기준은 [docs의 출제 PDF](../../docs/0820_DCC%20대학부_예선_출제문제.pdf) 5쪽, 9–12쪽입니다. 서울 Training만 학습하고 양측의 전사 `text`만 입력으로 사용합니다. `speaker`, 시간, 주소, 성별, 정답 라벨은 입력 특징에서 제외합니다. 대상은 `고열, 구토, 두통, 복통, 어지러움, 열상, 오심, 전신쇠약, 호흡곤란`이며, 나머지 라벨만 제거하고 통화는 유지합니다. 외부 추가 데이터와 상용 AI API를 사용하지 않습니다.\n\n최종 제출에는 추론 코드, 학습 로그 포함 노트북, 학습 완료 가중치, 모델 코드, requirements 및 결과 정리 PPT가 필요합니다. 이 구현의 학습을 완료하고 로그·가중치를 보관해야 하며 PPT는 별도로 준비합니다.\n\n공식 구현 참고: [KLUE RoBERTa large](https://huggingface.co/klue/roberta-large), [Transformers tokenizer](https://huggingface.co/docs/transformers/en/main_classes/tokenizer), [PyTorch AMP](https://docs.pytorch.org/docs/stable/notes/amp_examples.html). 모델 로딩·토큰 처리·혼합 정밀도 API 참고 자료이며 Mission 3의 성능 향상을 보장하는 근거는 아닙니다.\n', 'ver4_runtime/__init__.py': '"""Unmodified ver4 runtime retained for compatible inference and run verification."""\n', 'ver4_runtime/calibration.py': '"""Training-only grouped calibration for nine independent symptom labels."""\nfrom __future__ import annotations\n\nimport numpy as np\nfrom iterstrat.ml_stratifiers import MultilabelStratifiedKFold\nfrom sklearn.metrics import f1_score, precision_recall_fscore_support\n\nif __package__:\n    from .data import TARGET_SYMPTOMS\nelse:\n    from data import TARGET_SYMPTOMS\n\nWEIGHT_GRID = (0.0, 0.25, 0.5, 0.75, 1.0)\nALL_NEGATIVE_THRESHOLD = float(np.nextafter(1.0, np.inf))\n\n\ndef _probabilities(values, name):\n    values = np.asarray(values, dtype=np.float64)\n    if values.ndim != 2 or values.shape[1] != len(TARGET_SYMPTOMS):\n        raise ValueError(f"{name} must have shape (samples, {len(TARGET_SYMPTOMS)})")\n    if not len(values) or not np.isfinite(values).all():\n        raise ValueError(f"{name} must be nonempty and finite")\n    if np.any((values < 0) | (values > 1)):\n        raise ValueError(f"{name} must contain probabilities in [0, 1]")\n    return values\n\n\ndef _inputs(y_true, neural_probs, sparse_probs):\n    neural = _probabilities(neural_probs, "neural_probs")\n    sparse = _probabilities(sparse_probs, "sparse_probs")\n    truth = np.asarray(y_true)\n    if truth.shape != neural.shape or sparse.shape != neural.shape:\n        raise ValueError("truth, neural_probs and sparse_probs must have identical shapes")\n    if not np.isin(truth, (0, 1)).all():\n        raise ValueError("y_true must contain only binary labels")\n    return truth.astype(np.int8), neural, sparse\n\n\ndef _best_threshold(truth, probabilities):\n    """Exact F1 maximum for >= decisions; tied probabilities stay together."""\n    positives = int(truth.sum())\n    if positives == 0:\n        return ALL_NEGATIVE_THRESHOLD, 0.0\n    if positives == len(truth):\n        return 0.0, 1.0\n    order = np.argsort(-probabilities, kind="stable")\n    sorted_probs = probabilities[order]\n    # A threshold can include a complete tied block, never part of one.\n    boundaries = np.r_[np.flatnonzero(sorted_probs[:-1] != sorted_probs[1:]), len(truth) - 1]\n    true_positives = np.cumsum(truth[order])[boundaries]\n    scores = 2.0 * true_positives / (positives + boundaries + 1)\n    thresholds = np.zeros(len(boundaries), dtype=np.float64)\n    has_next = boundaries < len(truth) - 1\n    upper = sorted_probs[boundaries[has_next]]\n    lower = sorted_probs[boundaries[has_next] + 1]\n    midpoint = lower + (upper - lower) / 2\n    # Adjacent floating-point values may have no representable midpoint.\n    thresholds[has_next] = np.where(midpoint > lower, midpoint, upper)\n    best = np.flatnonzero(scores == scores.max())\n    # An explicit deterministic tie break preserves the exact maximum.\n    winner = min(best, key=lambda i: (abs(thresholds[i] - 0.5), -thresholds[i]))\n    return float(thresholds[winner]), float(scores[winner])\n\n\ndef fit_calibrator(y_true, neural_probs, sparse_probs):\n    """Select each label\'s blend and threshold using supplied training labels."""\n    truth, neural, sparse = _inputs(y_true, neural_probs, sparse_probs)\n    weights, thresholds = [], []\n    for column in range(len(TARGET_SYMPTOMS)):\n        candidates = []\n        for weight in WEIGHT_GRID:\n            blended = weight * neural[:, column] + (1.0 - weight) * sparse[:, column]\n            threshold, score = _best_threshold(truth[:, column], blended)\n            candidates.append((score, -abs(weight - 0.5), weight, threshold))\n        _, _, weight, threshold = max(candidates)\n        weights.append(float(weight))\n        thresholds.append(float(threshold))\n    return {"neural_weights": weights, "thresholds": thresholds}\n\n\ndef apply_calibrator(neural_probs, sparse_probs, calibrator):\n    """Return blended probabilities; use >= stored thresholds for labels."""\n    neural = _probabilities(neural_probs, "neural_probs")\n    sparse = _probabilities(sparse_probs, "sparse_probs")\n    if neural.shape != sparse.shape:\n        raise ValueError("neural_probs and sparse_probs must have identical shapes")\n    weights = np.asarray(calibrator["neural_weights"], dtype=np.float64)\n    thresholds = np.asarray(calibrator["thresholds"], dtype=np.float64)\n    expected = (len(TARGET_SYMPTOMS),)\n    if weights.shape != expected or thresholds.shape != expected:\n        raise ValueError("calibrator must contain one weight and threshold per symptom")\n    if not np.isfinite(weights).all() or np.any((weights < 0) | (weights > 1)):\n        raise ValueError("calibrator weights must be finite and in [0, 1]")\n    if not np.isfinite(thresholds).all() or np.any((thresholds < 0) | (thresholds > ALL_NEGATIVE_THRESHOLD)):\n        raise ValueError("calibrator thresholds must be finite and in [0, nextafter(1, inf)]")\n    return weights * neural + (1.0 - weights) * sparse\n\n\ndef _grouped_splits(truth, groups, seed, n_splits):\n    groups = np.asarray(groups)\n    if groups.ndim != 1 or len(groups) != len(truth):\n        raise ValueError("groups must contain one group ID per sample")\n    if not isinstance(n_splits, (int, np.integer)) or n_splits < 2:\n        raise ValueError("n_splits must be an integer >= 2")\n    unique_groups, inverse = np.unique(groups, return_inverse=True)\n    if len(unique_groups) < 2:\n        raise ValueError("cross-fit calibration requires at least two distinct groups")\n    n_splits = min(n_splits, len(unique_groups))\n    group_targets = np.zeros((len(unique_groups), truth.shape[1]), dtype=np.int8)\n    np.maximum.at(group_targets, inverse, truth)\n    splitter = MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)\n    for train_groups, test_groups in splitter.split(np.zeros(len(unique_groups)), group_targets):\n        train = np.flatnonzero(np.isin(inverse, train_groups))\n        test = np.flatnonzero(np.isin(inverse, test_groups))\n        if not len(train) or not len(test):\n            raise ValueError("grouped calibration produced an empty fold")\n        yield train, test\n\n\ndef calibrate_crossfit(y_true, neural_probs, sparse_probs, groups, seed=42, n_splits=5):\n    """Cross-fit within Training calibration data, then fit deployment settings.\n\n    Base-model predictions must themselves come from samples excluded from\n    base-model training and model selection. Never pass official Validation.\n    """\n    truth, neural, sparse = _inputs(y_true, neural_probs, sparse_probs)\n    crossfit_predictions = np.zeros_like(truth)\n    seen = np.zeros(len(truth), dtype=np.int8)\n    fold_counts = []\n    for train, test in _grouped_splits(truth, groups, seed, n_splits):\n        calibrator = fit_calibrator(truth[train], neural[train], sparse[train])\n        probabilities = apply_calibrator(neural[test], sparse[test], calibrator)\n        crossfit_predictions[test] = probabilities >= np.asarray(calibrator["thresholds"])\n        seen[test] += 1\n        fold_counts.append({"fit_samples": int(len(train)), "score_samples": int(len(test))})\n    if not np.all(seen == 1):\n        raise RuntimeError("each calibration sample must be scored exactly once")\n    final_calibrator = fit_calibrator(truth, neural, sparse)\n    fitted = apply_calibrator(neural, sparse, final_calibrator) >= np.asarray(final_calibrator["thresholds"])\n    precision, recall, f1, support = precision_recall_fscore_support(\n        truth, crossfit_predictions, average=None, zero_division=0\n    )\n    report = {\n        "score_scope": "Training-only calibration; cross-fit estimates calibration generalization, not official Validation.",\n        "calibration_fit_macro_f1": float(f1_score(truth, fitted, average="macro", zero_division=0)),\n        "calibration_crossfit_macro_f1": float(f1.mean()),\n        "per_symptom_crossfit": {\n            symptom: {"precision": float(p), "recall": float(r), "f1": float(f), "support": int(s)}\n            for symptom, p, r, f, s in zip(TARGET_SYMPTOMS, precision, recall, f1, support)\n        },\n        "calibration_samples": int(len(truth)),\n        "calibration_groups": int(len(np.unique(groups))),\n        "crossfit_splits": len(fold_counts),\n        "crossfit_fold_counts": fold_counts,\n    }\n    return final_calibrator, report\n', 'ver4_runtime/data.py': '"""Mission 3: 모델 입력은 양측 발화의 text만, symptom은 학습/평가 정답만."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport unicodedata\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport numpy as np\nfrom iterstrat.ml_stratifiers import MultilabelStratifiedKFold, MultilabelStratifiedShuffleSplit\nfrom sklearn.metrics import f1_score, precision_recall_fscore_support\n\nTARGET_SYMPTOMS = (\n    "고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란",\n)\n\n\ndef dialogue_text(annotation: dict) -> str:\n    """입력/정답과 무관하게 동일하게 전처리하며 발화 순서를 유지한다."""\n    utterances = annotation.get("utterances")\n    if not isinstance(utterances, list):\n        utterances = annotation.get("utterences", [])\n    if not isinstance(utterances, list):\n        raise ValueError("utterances/utterences must be a list")\n    texts = []\n    for utterance in utterances:\n        if isinstance(utterance, dict) and isinstance(utterance.get("text"), str):\n            text = " ".join(unicodedata.normalize("NFC", utterance["text"]).split())\n            if text:\n                texts.append(text)\n    return "\\n".join(texts)\n\n\ndef target_vector(annotation: dict) -> list[int]:\n    if "symptom" not in annotation:\n        raise ValueError("Training/evaluation JSON has no symptom target")\n    raw = annotation["symptom"]\n    if raw is None:\n        raw = []\n    if isinstance(raw, str):\n        raw = [raw]\n    if not isinstance(raw, list) or any(not isinstance(x, str) for x in raw):\n        raise ValueError("symptom must be a string or a list of strings")\n    found = {x.strip() for x in raw}\n    return [int(name in found) for name in TARGET_SYMPTOMS]\n\n\ndef text_group(text: str) -> str:\n    # 파일명, 날짜, speaker 등은 모델 특징에도 중복 판단에도 사용하지 않는다.\n    normalized = " ".join(unicodedata.normalize("NFC", text).split())\n    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()\n\n\ndef load_dataset(label_dir: Path, include_targets: bool = False, workers: int = 8):\n    paths = sorted(Path(label_dir).rglob("*.json"))\n    if not paths:\n        raise ValueError(f"JSON 파일이 없습니다: {label_dir}")\n    if len({p.name for p in paths}) != len(paths):\n        raise ValueError("동일한 JSON 파일명이 중복됩니다. 제출 CSV의 식별자가 모호합니다.")\n\n    def read(path):\n        with path.open(encoding="utf-8-sig") as handle:\n            annotation = json.load(handle)\n        if not isinstance(annotation, dict):\n            raise ValueError(f"JSON object가 아닙니다: {path}")\n        # 추론 경로에서는 target_vector를 호출하거나 symptom에 접근하지 않는다.\n        return dialogue_text(annotation), target_vector(annotation) if include_targets else None\n\n    with ThreadPoolExecutor(max_workers=workers) as executor:\n        rows = list(executor.map(read, paths))\n    texts = [row[0] for row in rows]\n    targets = np.asarray([row[1] for row in rows], dtype=np.int8) if include_targets else None\n    return paths, texts, targets\n\n\ndef grouped_targets(texts, targets):\n    groups = np.asarray([text_group(text) for text in texts])\n    unique, inverse = np.unique(groups, return_inverse=True)\n    aggregate = np.zeros((len(unique), len(TARGET_SYMPTOMS)), dtype=np.int8)\n    np.maximum.at(aggregate, inverse, targets)\n    return groups, inverse, aggregate\n\n\ndef make_splits(texts, targets, calibration_size: float, n_folds: int, seed: int):\n    """전체 Training의 보정 집합은 모든 모델 fit/epoch 선택에서 분리한다."""\n    if not 0 < calibration_size < 0.5 or n_folds < 2:\n        raise ValueError("calibration_size must be in (0, .5); n_folds >= 2")\n    groups, inverse, aggregate = grouped_targets(texts, targets)\n    group_ids = np.arange(len(aggregate))\n    outer = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=calibration_size, random_state=seed)\n    dev_groups, cal_groups = next(outer.split(group_ids, aggregate))\n    development = np.flatnonzero(np.isin(inverse, dev_groups))\n    calibration = np.flatnonzero(np.isin(inverse, cal_groups))\n    splitter = MultilabelStratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed + 1)\n    folds = []\n    for train_g, stop_g in splitter.split(dev_groups, aggregate[dev_groups]):\n        train_indices = np.flatnonzero(np.isin(inverse, dev_groups[train_g]))\n        stop_indices = np.flatnonzero(np.isin(inverse, dev_groups[stop_g]))\n        folds.append((train_indices, stop_indices))\n    for name, indices in [("calibration", calibration), ("development", development),\n                          *[(f"fold{i}/{part}", ids) for i, pair in enumerate(folds)\n                            for part, ids in zip(("fit", "stop"), pair)]]:\n        counts = targets[indices].sum(axis=0)\n        if len(indices) == 0 or np.any(counts == 0) or np.any(counts == len(indices)):\n            raise ValueError(f"{name}: 각 증상의 양성/음성이 필요합니다. 데이터/분할 수를 확인하세요.")\n    return development, calibration, folds, groups\n\n\ndef write_json(path: Path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef write_predictions(paths, probabilities, thresholds, output: Path):\n    probabilities = np.asarray(probabilities)\n    if probabilities.shape != (len(paths), len(TARGET_SYMPTOMS)) or not np.isfinite(probabilities).all():\n        raise ValueError("Invalid prediction shape or non-finite probabilities")\n    predicted = probabilities >= np.asarray(thresholds)\n    output = Path(output)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with output.open("w", encoding="utf-8-sig", newline="") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["label file name", "symptom"])\n        for path, flags in zip(paths, predicted):\n            # 기존 제출 CSV와 같은 Python string-list 표현. 빈 list도 허용한다.\n            writer.writerow([Path(path).name, str([s for s, flag in zip(TARGET_SYMPTOMS, flags) if flag])])\n    return predicted.astype(np.int8)\n\n\ndef score_predictions(targets, predictions):\n    precision, recall, f1, support = precision_recall_fscore_support(targets, predictions, zero_division=0)\n    return {\n        "macro_f1": float(f1_score(targets, predictions, average="macro", zero_division=0)),\n        "samples": int(len(targets)),\n        "per_symptom": {name: {"f1": float(f1[i]), "precision": float(precision[i]),\n                                "recall": float(recall[i]), "support": int(support[i])}\n                        for i, name in enumerate(TARGET_SYMPTOMS)},\n    }\n', 'ver4_runtime/evaluate.py': '"""Score frozen CSV predictions after inference, without changing the model."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nif __package__:\n    from .data import write_json\n    from .pipeline import evaluate_csv\nelse:\n    from data import write_json\n    from pipeline import evaluate_csv\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--label_dir", required=True, type=Path)\n    parser.add_argument("--prediction", required=True, type=Path)\n    parser.add_argument("--output", type=Path)\n    args = parser.parse_args()\n    report = evaluate_csv(args.label_dir, args.prediction)\n    if args.output:\n        write_json(args.output, report)\n    print(json.dumps(report, ensure_ascii=False, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'ver4_runtime/inference.py': '"""Official one-command inference. Mission 3 accepts but does not read audio_dir."""\nimport argparse\nfrom pathlib import Path\n\nif __package__:\n    from .pipeline import predict_pipeline\nelse:\n    from pipeline import predict_pipeline\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--audio_dir", required=True, help="Organizer interface; unused text-only task")\n    parser.add_argument("--label_dir", required=True, type=Path)\n    parser.add_argument("--ckpt_path", required=True, type=Path)\n    parser.add_argument("--output", required=True, type=Path)\n    parser.add_argument("--device", choices=["cpu", "cuda"])\n    parser.add_argument("--batch_size", type=int)\n    args = parser.parse_args()\n    predict_pipeline(args.ckpt_path, args.label_dir, args.output, args.device, args.batch_size)\n\n\nif __name__ == "__main__":\n    main()\n', 'ver4_runtime/neural.py': '"""Document-level multi-label transformer; every dialogue window is retained."""\n\nfrom __future__ import annotations\n\nimport math\nimport os\nimport random\nimport re\nimport tempfile\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom sklearn.metrics import average_precision_score\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Dataset\nfrom transformers import AutoConfig, AutoModel, get_linear_schedule_with_warmup\n\n\nNUM_LABELS = 9\n\n\ndef tokenize_dialogues(texts, tokenizer, max_length=512, stride=64):\n    """Return documents of token windows, including the last window of long calls.\n\n    Only ordinary tokenizer special tokens are used. The stride is an overlap in\n    tokens, not an upper bound on the number of windows or dialogue length.\n    """\n    usable_length = max_length - tokenizer.num_special_tokens_to_add(pair=False)\n    if usable_length < 1 or not 0 <= stride < usable_length:\n        raise ValueError("max_length must fit special tokens; 0 <= stride < usable length")\n    texts = list(texts)\n    documents = [[] for _ in texts]\n    for start in range(0, len(texts), 128):\n        encoded = tokenizer(\n            texts[start:start + 128], truncation=True, max_length=max_length,\n            stride=stride, return_overflowing_tokens=True, padding=False,\n            return_attention_mask=True,\n        )\n        if "overflow_to_sample_mapping" not in encoded:\n            raise ValueError("Use a fast tokenizer (AutoTokenizer(..., use_fast=True))")\n        for window, sample in enumerate(encoded["overflow_to_sample_mapping"]):\n            documents[start + int(sample)].append({\n                key: encoded[key][window]\n                for key in tokenizer.model_input_names if key in encoded\n            })\n    if any(not windows for windows in documents):\n        raise ValueError("Tokenizer returned a document without any windows")\n    return documents\n\n\nclass DialogueDataset(Dataset):\n    def __init__(self, documents, targets=None):\n        self.documents, self.targets = documents, targets\n\n    def __len__(self):\n        return len(self.documents)\n\n    def __getitem__(self, index):\n        item = {"windows": self.documents[index]}\n        if self.targets is not None:\n            item["labels"] = self.targets[index]\n        return item\n\n\nclass DialogueCollator:\n    """Flatten windows for the encoder while keeping exactly one target per call."""\n\n    def __init__(self, tokenizer):\n        self.tokenizer = tokenizer\n\n    def __call__(self, examples):\n        windows, doc_indices = [], []\n        for index, example in enumerate(examples):\n            if not example["windows"]:\n                raise ValueError("A document must contain at least one token window")\n            windows.extend(example["windows"])\n            doc_indices.extend([index] * len(example["windows"]))\n        batch = dict(self.tokenizer.pad(windows, padding=True, return_tensors="pt"))\n        batch["doc_indices"] = torch.tensor(doc_indices, dtype=torch.long)\n        if "labels" in examples[0]:\n            batch["labels"] = torch.from_numpy(np.asarray(\n                [example["labels"] for example in examples], dtype=np.float32,\n            ))\n        return batch\n\n\nclass DialogueClassifier(nn.Module):\n    """Label-specific attention pools complete dialogues before computing BCE."""\n\n    def __init__(self, encoder, num_labels=NUM_LABELS, dropout=0.2,\n                 window_batch_size=8):\n        super().__init__()\n        self.encoder = encoder\n        self.window_batch_size = int(window_batch_size)\n        if self.window_batch_size < 1:\n            raise ValueError("window_batch_size must be positive")\n        hidden = encoder.config.hidden_size\n        self.projection = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.Tanh())\n        self.dropout = nn.Dropout(dropout)\n        self.attention_queries = nn.Parameter(torch.empty(num_labels, hidden))\n        self.label_weights = nn.Parameter(torch.empty(num_labels, hidden))\n        self.label_bias = nn.Parameter(torch.zeros(num_labels))\n        nn.init.normal_(self.attention_queries, std=0.02)\n        nn.init.normal_(self.label_weights, std=0.02)\n\n    def forward(self, doc_indices, **inputs):\n        representations = []\n        for start in range(0, len(doc_indices), self.window_batch_size):\n            window_inputs = {key: value[start:start + self.window_batch_size]\n                             for key, value in inputs.items()}\n            hidden = self.encoder(**window_inputs).last_hidden_state\n            mask = window_inputs["attention_mask"].unsqueeze(-1).to(hidden.dtype)\n            mean = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)\n            representations.append(self.projection(torch.cat([hidden[:, 0], mean], -1)))\n        windows = self.dropout(torch.cat(representations))\n        logits = []\n        # Each label learns which part of a potentially long call provides evidence.\n        for document_index in range(int(doc_indices.max().item()) + 1):\n            selected = windows[doc_indices == document_index]\n            scores = selected @ self.attention_queries.T / math.sqrt(selected.shape[-1])\n            attention = torch.softmax(scores.float(), dim=0).to(selected.dtype)\n            pooled = attention.T @ selected\n            logits.append((self.dropout(pooled) * self.label_weights).sum(-1) + self.label_bias)\n        return torch.stack(logits)\n\n\ndef _device(config):\n    device = torch.device(config.get("device", "cuda" if torch.cuda.is_available() else "cpu"))\n    if device.type == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("CUDA was requested but is unavailable. Select a GPU runtime.")\n    return device\n\n\ndef _amp(device):\n    dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16\n    return torch.amp.autocast(device_type=device.type, dtype=dtype, enabled=device.type == "cuda")\n\n\ndef _loader(documents, targets, tokenizer, batch_size, shuffle=False, seed=0):\n    if int(batch_size) < 1:\n        raise ValueError("batch_size must be positive")\n    generator = torch.Generator().manual_seed(seed)\n    return DataLoader(\n        DialogueDataset(documents, targets), batch_size=int(batch_size),\n        shuffle=shuffle, collate_fn=DialogueCollator(tokenizer),\n        num_workers=0, generator=generator,\n    )\n\n\n@torch.inference_mode()\ndef _predict(model, loader, device):\n    model.eval()\n    probabilities = []\n    for batch in loader:\n        batch.pop("labels", None)\n        batch = {key: value.to(device) for key, value in batch.items()}\n        with _amp(device):\n            logits = model(**batch)\n        probabilities.append(logits.float().sigmoid().cpu().numpy())\n    return np.concatenate(probabilities) if probabilities else np.empty((0, NUM_LABELS), dtype=np.float32)\n\n\ndef _optimizer(model, config):\n    depth = int(getattr(model.encoder.config, "num_hidden_layers", 0))\n    decay = float(config.get("layerwise_lr_decay", 0.9))\n    if not 0 < decay <= 1:\n        raise ValueError("layerwise_lr_decay must be in (0, 1]")\n    groups = {}\n    for name, parameter in model.named_parameters():\n        if not parameter.requires_grad:\n            continue\n        if name.startswith("encoder."):\n            match = re.search(r"\\.layer\\.(\\d+)\\.", name)\n            level = int(match.group(1)) + 1 if match else (0 if ".embeddings." in name else depth + 1)\n            lr = float(config.get("learning_rate", 1.5e-5)) * decay ** (depth + 1 - level)\n        else:\n            lr = float(config.get("head_learning_rate", 1e-4))\n        wd = 0.0 if parameter.ndim < 2 or name.endswith("bias") else float(config.get("weight_decay", 0.01))\n        groups.setdefault((lr, wd), []).append(parameter)\n    return torch.optim.AdamW([{"params": params, "lr": lr, "weight_decay": wd}\n                             for (lr, wd), params in groups.items()])\n\n\ndef _atomic_save(checkpoint, path):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    descriptor, temporary = tempfile.mkstemp(prefix=path.name + ".", suffix=".tmp", dir=path.parent)\n    os.close(descriptor)\n    try:\n        torch.save(checkpoint, temporary)\n        os.replace(temporary, path)\n    finally:\n        if os.path.exists(temporary):\n            os.unlink(temporary)\n\n\ndef _targets(targets, documents, name):\n    targets = np.asarray(targets, dtype=np.float32)\n    if targets.shape != (len(documents), NUM_LABELS) or not len(documents):\n        raise ValueError(f"{name} requires nonempty documents and targets of shape (N, {NUM_LABELS})")\n    if not np.isin(targets, [0, 1]).all():\n        raise ValueError(f"{name} targets must contain only 0 and 1")\n    return targets\n\n\ndef train_model(train_documents, train_targets, stop_documents, stop_targets,\n                tokenizer, config: dict, checkpoint_path: Path, seed: int):\n    """Train on documents; select best epoch only by stopping-set macro AP.\n\n    This function does not tune thresholds and never uses calibration targets.\n    GPU enforcement for production belongs to the orchestration entry point.\n    """\n    train_targets = _targets(train_targets, train_documents, "train")\n    stop_targets = _targets(stop_targets, stop_documents, "stop")\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    device = _device(config)\n    max_epochs, patience = int(config.get("max_epochs", 8)), int(config.get("patience", 2))\n    accumulation = int(config.get("gradient_accumulation_steps", 4))\n    if min(max_epochs, patience, accumulation) < 1:\n        raise ValueError("max_epochs, patience and gradient_accumulation_steps must be positive")\n    model_name = config.get("model_name", "klue/roberta-large")\n    encoder = AutoModel.from_pretrained(model_name)\n    if config.get("gradient_checkpointing", True):\n        encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n    if hasattr(encoder.config, "use_cache"):\n        encoder.config.use_cache = False\n    architecture = {"num_labels": NUM_LABELS, "dropout": float(config.get("dropout", 0.2)),\n                    "window_batch_size": int(config.get("window_batch_size", 8))}\n    model = DialogueClassifier(encoder, **architecture).to(device)\n    train_loader = _loader(train_documents, train_targets, tokenizer,\n                           config.get("batch_size", 2), shuffle=True, seed=seed)\n    stop_loader = _loader(stop_documents, None, tokenizer, config.get("inference_batch_size", 4))\n    positives = train_targets.sum(axis=0)\n    if (positives == 0).any():\n        raise ValueError(f"Training split has no positive examples for labels {np.flatnonzero(positives == 0).tolist()}")\n    pos_weight = np.clip(((len(train_targets) - positives) / positives).clip(min=1)\n                         ** float(config.get("pos_weight_power", 0.5)), 1, 5)\n    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.as_tensor(pos_weight, device=device))\n    optimizer = _optimizer(model, config)\n    total_steps = math.ceil(len(train_loader) / accumulation) * max_epochs\n    scheduler = get_linear_schedule_with_warmup(\n        optimizer, num_warmup_steps=int(total_steps * float(config.get("warmup_ratio", 0.1))),\n        num_training_steps=total_steps,\n    )\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and not torch.cuda.is_bf16_supported())\n    valid_labels = stop_targets.sum(axis=0) > 0\n    if not valid_labels.any():\n        raise ValueError("Stopping split has no positive labels; macro AP cannot select an epoch")\n    if not valid_labels.all():\n        print(f"Warning: stop AP excludes labels without positives: {np.flatnonzero(~valid_labels).tolist()}", flush=True)\n    best_ap, best_epoch, stale, history = -1.0, 0, 0, []\n    for epoch in range(1, max_epochs + 1):\n        epoch_started = time.monotonic()\n        model.train()\n        optimizer.zero_grad(set_to_none=True)\n        loss_sum, document_count = 0.0, 0\n        for step, batch in enumerate(train_loader):\n            labels = batch.pop("labels").to(device)\n            batch = {key: value.to(device) for key, value in batch.items()}\n            # Match a single full document batch, including a shorter last batch.\n            group_start = (step // accumulation) * accumulation\n            group_documents = min(accumulation * train_loader.batch_size,\n                                  len(train_documents) - group_start * train_loader.batch_size)\n            with _amp(device):\n                loss = criterion(model(**batch).float(), labels)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(f"Non-finite training loss at epoch={epoch}, batch={step}")\n            scaler.scale(loss * len(labels) / group_documents).backward()\n            loss_sum += loss.detach().item() * len(labels)\n            document_count += len(labels)\n            if (step + 1) % accumulation == 0 or step + 1 == len(train_loader):\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                old_scale = scaler.get_scale()\n                scaler.step(optimizer)\n                scaler.update()\n                if scaler.get_scale() >= old_scale:\n                    scheduler.step()\n                optimizer.zero_grad(set_to_none=True)\n            log_every = int(config.get("log_every", 100))\n            if log_every > 0 and (step + 1) % log_every == 0:\n                elapsed = (time.monotonic() - epoch_started) / 60\n                print(f"seed={seed} epoch={epoch} batch={step + 1}/{len(train_loader)} "\n                      f"loss={loss_sum / document_count:.5f} minutes={elapsed:.1f}", flush=True)\n        probabilities = _predict(model, stop_loader, device)\n        if not np.isfinite(probabilities).all():\n            raise FloatingPointError("Stopping predictions contain non-finite values")\n        score = float(average_precision_score(stop_targets[:, valid_labels],\n                                              probabilities[:, valid_labels], average="macro"))\n        row = {"epoch": epoch, "train_loss": loss_sum / document_count,\n               "stop_macro_ap": score, "stop_labels_with_positives": int(valid_labels.sum()),\n               "minutes": (time.monotonic() - epoch_started) / 60}\n        history.append(row)\n        print(f"seed={seed} epoch={epoch}/{max_epochs} loss={row[\'train_loss\']:.5f} "\n              f"stop_AP={score:.5f} minutes={row[\'minutes\']:.1f}", flush=True)\n        if score > best_ap + 1e-6:\n            best_ap, best_epoch, stale = score, epoch, 0\n            _atomic_save({\n                "format_version": 1, "encoder_config": encoder.config.to_dict(),\n                "architecture": architecture, "model_name": str(model_name),\n                "model_state": {key: value.detach().cpu() for key, value in model.state_dict().items()},\n                "epoch": epoch, "stop_ap": score, "seed": int(seed),\n            }, checkpoint_path)\n        else:\n            stale += 1\n            if stale >= patience:\n                break\n    return {"best_epoch": best_epoch, "best_stop_ap": best_ap, "history": history}\n\n\ndef predict_checkpoint(checkpoint_path, documents, tokenizer, config):\n    """Rebuild from the saved config and weights; no pretrained network fetch."""\n    device = _device(config)\n    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    if checkpoint.get("format_version") != 1:\n        raise ValueError(f"Unsupported neural checkpoint format: {checkpoint_path}")\n    encoder_data = dict(checkpoint["encoder_config"])\n    model_type = encoder_data.pop("model_type")\n    encoder_config = AutoConfig.for_model(model_type, **encoder_data)\n    architecture = dict(checkpoint["architecture"])\n    architecture["window_batch_size"] = int(config.get("window_batch_size", architecture["window_batch_size"]))\n    model = DialogueClassifier(AutoModel.from_config(encoder_config), **architecture)\n    model.load_state_dict(checkpoint["model_state"], strict=True)\n    del checkpoint\n    model.to(device)\n    loader = _loader(documents, None, tokenizer, config.get("inference_batch_size", 4))\n    predictions = _predict(model, loader, device)\n    if not np.isfinite(predictions).all():\n        raise FloatingPointError("Inference predictions contain non-finite values")\n    return predictions\n', 'ver4_runtime/pipeline.py': '"""Training-only model selection, ensemble calibration, and portable inference."""\nfrom __future__ import annotations\n\nimport ast\nimport csv\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport shutil\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport torch\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import FeatureUnion\nfrom transformers import AutoTokenizer\n\nif __package__:\n    from .calibration import apply_calibrator, calibrate_crossfit\n    from .data import TARGET_SYMPTOMS, load_dataset, make_splits, score_predictions, write_json, write_predictions\n    from .neural import predict_checkpoint, tokenize_dialogues, train_model\nelse:\n    from calibration import apply_calibrator, calibrate_crossfit\n    from data import TARGET_SYMPTOMS, load_dataset, make_splits, score_predictions, write_json, write_predictions\n    from neural import predict_checkpoint, tokenize_dialogues, train_model\n\n\nDEFAULT_CONFIG = {\n    "model_name": "klue/roberta-large",\n    "seed": 42,\n    "n_folds": 5,\n    "calibration_size": 0.10,\n    "calibration_folds": 5,\n    "max_length": 512,\n    "stride": 64,\n    "max_epochs": 6,\n    "patience": 2,\n    "batch_size": 2,\n    "inference_batch_size": 8,\n    "window_batch_size": 2,\n    "gradient_accumulation_steps": 16,\n    "learning_rate": 1.5e-5,\n    "head_learning_rate": 1e-4,\n    "layerwise_lr_decay": 0.95,\n    "weight_decay": 0.01,\n    "warmup_ratio": 0.10,\n    "dropout": 0.20,\n    "pos_weight_power": 0.5,\n    "gradient_checkpointing": True,\n    "device": "cuda",\n    "require_cuda": True,\n    "char_features": 120_000,\n    "word_features": 40_000,\n    "min_df": 2,\n    "logistic_c": 4.0,\n}\nRUNTIME_FILES = ("data.py", "calibration.py", "neural.py", "pipeline.py", "train.py", "inference.py", "evaluate.py")\n\n\ndef fit_sparse(texts, targets, config):\n    """문자 표현과 단어 순서를 보완하는, Training에만 fit한 TF-IDF 모델."""\n    features = FeatureUnion([\n        ("char", TfidfVectorizer(analyzer="char", ngram_range=(2, 5),\n                                max_features=config["char_features"], min_df=config["min_df"],\n                                sublinear_tf=True, dtype=np.float32)),\n        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), token_pattern=r"(?u)\\b\\w+\\b",\n                                max_features=config["word_features"], min_df=config["min_df"],\n                                sublinear_tf=True, dtype=np.float32)),\n    ])\n    matrix = features.fit_transform(texts)\n    classifiers = []\n    for column, name in enumerate(TARGET_SYMPTOMS):\n        truth = targets[:, column]\n        if len(np.unique(truth)) == 1:\n            classifier = float(truth[0])\n        else:\n            classifier = LogisticRegression(C=config["logistic_c"], class_weight="balanced",\n                                            solver="liblinear", max_iter=1500, random_state=config["seed"])\n            classifier.fit(matrix, truth)\n        classifiers.append(classifier)\n        print(f"TF-IDF fit: {name}", flush=True)\n    return {"features": features, "classifiers": classifiers}\n\n\ndef predict_sparse(model, texts):\n    if not texts:\n        return np.empty((0, len(TARGET_SYMPTOMS)), dtype=np.float32)\n    # 큰 Test 폴더도 전체 sparse matrix를 RAM에 동시에 올리지 않는다.\n    batches = []\n    for start in range(0, len(texts), 1024):\n        matrix = model["features"].transform(texts[start:start + 1024])\n        columns = [np.full(matrix.shape[0], clf) if isinstance(clf, float)\n                   else clf.predict_proba(matrix)[:, 1] for clf in model["classifiers"]]\n        batches.append(np.column_stack(columns))\n    return np.concatenate(batches).astype(np.float32)\n\n\ndef _release_memory():\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\ndef _torch_save(path, value):\n    path = Path(path)\n    temporary = path.with_name(path.name + ".tmp")\n    torch.save(value, temporary)\n    temporary.replace(path)\n\n\ndef _fingerprint(paths, texts, targets, config):\n    digest = hashlib.sha256(json.dumps(config, sort_keys=True).encode())\n    for path, text, target in zip(paths, texts, targets):\n        digest.update(json.dumps([path.name, text, target.tolist()], ensure_ascii=False).encode("utf-8"))\n    # 변경된 코드로 기존 fold가 조용히 재사용되는 것을 방지한다.\n    for name in RUNTIME_FILES:\n        path = Path(__file__).parent / name\n        digest.update(path.name.encode())\n        digest.update(path.read_bytes())\n    return digest.hexdigest()\n\n\ndef train_pipeline(train_label_dir, output_dir, config=None):\n    """Validation 인자를 받지 않는다. 보정 후에도 모델을 재학습하지 않는다."""\n    supplied = config or {}\n    unknown = set(supplied) - set(DEFAULT_CONFIG)\n    if unknown:\n        raise ValueError(f"Unknown config keys: {sorted(unknown)}")\n    config = {**DEFAULT_CONFIG, **supplied}\n    train_label_dir, output_dir = Path(train_label_dir).resolve(), Path(output_dir).resolve()\n    if train_label_dir.name != "TL_서울_구급" or "Validation" in train_label_dir.parts:\n        raise ValueError("학습 입력은 서울 Training의 TL_서울_구급 폴더여야 합니다.")\n    if config["require_cuda"] and (config["device"] != "cuda" or not torch.cuda.is_available()):\n        raise RuntimeError("전체 학습에는 GPU가 필요합니다. Colab GPU 런타임에서 실행하세요.")\n    if config["device"] == "cuda":\n        torch.set_float32_matmul_precision("high")\n        print("GPU:", torch.cuda.get_device_name(0), flush=True)\n    paths, texts, targets = load_dataset(train_label_dir, include_targets=True)\n    dev, calibration, folds, groups = make_splits(\n        texts, targets, config["calibration_size"], config["n_folds"], config["seed"],\n    )\n    output_dir.mkdir(parents=True, exist_ok=True)\n    signature = _fingerprint(paths, texts, targets, config)\n    run_path = output_dir / "run.json"\n    if run_path.exists() and json.loads(run_path.read_text(encoding="utf-8"))["signature"] != signature:\n        raise ValueError("출력 폴더에 다른 데이터/설정/코드의 실험이 있습니다. 새 output_dir를 지정하세요.")\n    write_json(run_path, {"signature": signature, "config": config, "training_samples": len(paths)})\n    write_json(output_dir / "splits.json", {\n        "calibration": [paths[i].name for i in calibration],\n        "folds": [{"fit": [paths[i].name for i in fit], "stop": [paths[i].name for i in stop]}\n                  for fit, stop in folds],\n        "seed": config["seed"], "grouping": "SHA256 of whitespace-normalized transcript only",\n    })\n    print(f"Training={len(paths):,}, development={len(dev):,}, calibration={len(calibration):,}", flush=True)\n    tokenizer_dir = output_dir / "tokenizer"\n    if (tokenizer_dir / "tokenizer_config.json").exists():\n        tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir, use_fast=True, local_files_only=True)\n    else:\n        tokenizer = AutoTokenizer.from_pretrained(config["model_name"], use_fast=True)\n        tokenizer.save_pretrained(tokenizer_dir)\n    documents = tokenize_dialogues(texts, tokenizer, config["max_length"], config["stride"])\n    calibration_documents = [documents[i] for i in calibration]\n    counts = np.asarray([len(document) for document in documents])\n    print(f"대화 window 수: 평균={counts.mean():.2f}, 최대={counts.max()}, 긴 대화={int((counts > 1).sum()):,}", flush=True)\n    neural_probs = np.zeros((len(calibration), len(TARGET_SYMPTOMS)), dtype=np.float64)\n    fold_names, histories = [], []\n    for number, (fit, stop) in enumerate(folds, 1):\n        name = f"fold_{number}.ckpt"\n        checkpoint_path, done_path = output_dir / name, output_dir / f"fold_{number}_done.json"\n        print(f"\\nFold {number}/{len(folds)}: fit={len(fit):,}, stop={len(stop):,}", flush=True)\n        if done_path.exists() and checkpoint_path.exists():\n            result = json.loads(done_path.read_text(encoding="utf-8"))\n            print("완료된 fold 재사용 (동일 run signature 확인됨)", flush=True)\n        else:\n            result = train_model([documents[i] for i in fit], targets[fit],\n                                 [documents[i] for i in stop], targets[stop], tokenizer,\n                                 config, checkpoint_path, seed=config["seed"] + number * 1009)\n            write_json(done_path, result)\n        _release_memory()\n        neural_probs += predict_checkpoint(checkpoint_path, calibration_documents, tokenizer, config)\n        _release_memory()\n        fold_names.append(name)\n        histories.append(result)\n    neural_probs /= len(folds)\n    sparse_path = output_dir / "sparse.joblib"\n    if sparse_path.exists():\n        sparse = joblib.load(sparse_path)\n    else:\n        sparse = fit_sparse([texts[i] for i in dev], targets[dev], config)\n        temporary = sparse_path.with_name("sparse.joblib.tmp")\n        joblib.dump(sparse, temporary, compress=3)\n        temporary.replace(sparse_path)\n    sparse_probs = predict_sparse(sparse, [texts[i] for i in calibration])\n    del sparse\n    calibrator, report = calibrate_crossfit(targets[calibration], neural_probs, sparse_probs,\n                                          groups[calibration], seed=config["seed"] + 2,\n                                          n_splits=config["calibration_folds"])\n    # OOF 확률을 final refit에 전용하지 않고, 실제 배포 ensemble의 확률을 보정한다.\n    np.savez_compressed(output_dir / "calibration_predictions.npz", targets=targets[calibration],\n                        neural=neural_probs, sparse=sparse_probs,\n                        filenames=np.asarray([paths[i].name for i in calibration]))\n    versions = {name: importlib.metadata.version(name) for name in\n                ("torch", "transformers", "numpy", "scikit-learn", "iterative-stratification")}\n    manifest = {"format": "mission3_ver4", "version": 1, "target_symptoms": list(TARGET_SYMPTOMS),\n                "config": config, "folds": fold_names, "tokenizer": "tokenizer", "sparse": "sparse.joblib",\n                "calibrator": calibrator, "signature": signature, "versions": versions}\n    checkpoint_path = output_dir / "mission3_ver4_model.ckpt"\n    _torch_save(checkpoint_path, manifest)\n    report.update({"training_samples": len(paths), "development_samples": len(dev),\n                   "model": f"{config[\'model_name\']} {len(folds)}-fold ensemble + char/word TF-IDF LR",\n                   "calibrator": calibrator, "fold_best_epochs": [x["best_epoch"] for x in histories],\n                   "versions": versions, "validation_used_for_training_or_calibration": False})\n    write_json(output_dir / "mission3_ver4_metrics.json", report)\n    write_json(output_dir / "mission3_ver4_history.json", histories)\n    # 노트북 하나만 Colab에 올려도 제출 코드와 실행 당시 설정을 남긴다.\n    submission = output_dir / "submission"\n    submission.mkdir(exist_ok=True)\n    for name in (*RUNTIME_FILES, "requirements.txt", "README.md"):\n        path = Path(__file__).parent / name\n        if path.exists():\n            shutil.copy2(path, submission / path.name)\n    print(json.dumps(report, ensure_ascii=False, indent=2), flush=True)\n    print("Checkpoint:", checkpoint_path, flush=True)\n    return checkpoint_path, report\n\n\ndef load_manifest(checkpoint_path):\n    manifest = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    if manifest.get("format") != "mission3_ver4" or manifest.get("version") != 1:\n        raise ValueError("지원하지 않는 checkpoint입니다. ver4의 manifest .ckpt를 지정하세요.")\n    if manifest["target_symptoms"] != list(TARGET_SYMPTOMS) or not manifest["folds"]:\n        raise ValueError("Checkpoint class order/folds mismatch")\n    root = Path(checkpoint_path).resolve().parent\n    for name in [*manifest["folds"], manifest["sparse"], manifest["tokenizer"]]:\n        path = (root / name).resolve()\n        if not path.is_relative_to(root) or not path.exists():\n            raise FileNotFoundError(f"Checkpoint 구성 파일을 함께 보관해야 합니다: {name}")\n    return manifest\n\n\ndef predict_pipeline(checkpoint_path, label_dir, output, device=None, batch_size=None):\n    """저장된 encoder config/tokenizer만 사용하므로 학습 후 오프라인 추론 가능."""\n    checkpoint_path = Path(checkpoint_path).resolve()\n    manifest = load_manifest(checkpoint_path)\n    root = checkpoint_path.parent\n    config = dict(manifest["config"])\n    config["device"] = device or ("cuda" if torch.cuda.is_available() else "cpu")\n    if batch_size is not None:\n        config["inference_batch_size"] = batch_size\n    paths, texts, _ = load_dataset(Path(label_dir), include_targets=False)\n    tokenizer = AutoTokenizer.from_pretrained(root / manifest["tokenizer"], use_fast=True, local_files_only=True)\n    documents = tokenize_dialogues(texts, tokenizer, config["max_length"], config["stride"])\n    neural_probs = np.zeros((len(paths), len(TARGET_SYMPTOMS)), dtype=np.float64)\n    for name in manifest["folds"]:\n        print("Predict:", name, flush=True)\n        neural_probs += predict_checkpoint(root / name, documents, tokenizer, config)\n        _release_memory()\n    neural_probs /= len(manifest["folds"])\n    sparse = joblib.load(root / manifest["sparse"])\n    sparse_probs = predict_sparse(sparse, texts)\n    probabilities = apply_calibrator(neural_probs, sparse_probs, manifest["calibrator"])\n    predictions = write_predictions(paths, probabilities, manifest["calibrator"]["thresholds"], Path(output))\n    print(f"{len(paths):,} predictions saved: {output}", flush=True)\n    return paths, probabilities, predictions\n\n\ndef evaluate_csv(label_dir, prediction_path):\n    """이미 저장된 예측을 파일명으로 정답에 연결한다. 모델/임계값 변경 없음."""\n    with Path(prediction_path).open(encoding="utf-8-sig", newline="") as handle:\n        reader = csv.DictReader(handle)\n        if reader.fieldnames != ["label file name", "symptom"]:\n            raise ValueError("CSV header must be: label file name,symptom")\n        rows = list(reader)\n    by_name = {}\n    for row in rows:\n        if row["label file name"] in by_name:\n            raise ValueError("Duplicate prediction filename")\n        labels = ast.literal_eval(row["symptom"])\n        if not isinstance(labels, list) or any(x not in TARGET_SYMPTOMS for x in labels) or len(set(labels)) != len(labels):\n            raise ValueError("CSV symptom must be a list of unique target strings")\n        by_name[row["label file name"]] = [int(x in labels) for x in TARGET_SYMPTOMS]\n    paths, _, targets = load_dataset(Path(label_dir), include_targets=True)\n    if set(by_name) != {path.name for path in paths}:\n        raise ValueError("CSV와 JSON 파일명 집합이 일치하지 않습니다.")\n    return score_predictions(targets, np.asarray([by_name[path.name] for path in paths], dtype=np.int8))\n', 'ver4_runtime/requirements.txt': 'torch>=2.6,<3\ntransformers==4.57.1\nnumpy>=1.26,<3\nscipy>=1.11,<2\nscikit-learn>=1.4,<2\niterative-stratification==0.1.9\njoblib>=1.3,<2\n', 'ver4_runtime/train.py': '"""Run all Training-only fitting and calibration; never accepts Validation."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nif __package__:\n    from .pipeline import train_pipeline\nelse:\n    from pipeline import train_pipeline\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--label_dir", required=True, type=Path)\n    parser.add_argument("--output_dir", required=True, type=Path)\n    parser.add_argument("--config", type=Path, help="Optional JSON overrides of DEFAULT_CONFIG")\n    args = parser.parse_args()\n    config = json.loads(args.config.read_text(encoding="utf-8")) if args.config else None\n    train_pipeline(args.label_dir, args.output_dir, config)\n\n\nif __name__ == "__main__":\n    main()\n'}
runtime_dir = Path(tempfile.mkdtemp(prefix="mission3_ver7_runtime_"))
for name, source in RUNTIME_SOURCES.items():
    path = runtime_dir / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding="utf-8")
config_path = runtime_dir / "notebook_config.json"
config_path.write_text(json.dumps(TRAIN_CONFIG, ensure_ascii=False, indent=2), encoding="utf-8")

def _run_python(arguments, allow_download=False):
    command = [sys.executable, "-u", *map(str, arguments)]
    offline = "0" if allow_download else "1"
    environment = dict(os.environ, HF_HUB_OFFLINE=offline, TRANSFORMERS_OFFLINE=offline,
                       HF_HUB_DISABLE_TELEMETRY="1", TOKENIZERS_PARALLELISM="false")
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, encoding="utf-8", errors="replace", env=environment)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        return_code = process.wait()
    except BaseException:
        process.kill()
        process.wait()
        raise
    finally:
        process.stdout.close()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

def run_python(arguments):
    return _run_python(arguments)

def run_pretrained():
    # 공개 모델 다운로드에만 네트워크 허용. 대회 입력 경로는 전달하지 않습니다.
    return _run_python([runtime_dir / "pretrained.py", "--destination", PRETRAINED_DIR], allow_download=True)

print("실행 코드:", runtime_dir)


In [ ]:
# 5. 공개 사전학습 encoder 자동 다운로드·검증 (같은 런타임에서는 검증 후 재사용)
download_started = time.monotonic()
run_pretrained()
PRETRAINED_MINUTES = (time.monotonic() - download_started) / 60
print(f"사전학습 모델 준비: {PRETRAINED_MINUTES:.1f}분")


## 6. 새로운 encoder 학습 → Training 내부 비교·선택

발화 순서를 유지하며 모든 발화 text를 같은 `[SEP]`로 구분합니다. speaker·증상 정답에 따라 입력을 바꾸지 않습니다.
긴 대화는 window로 나누고, token 표현에 9개 증상별 attention을 적용합니다.
**KoELECTRA 전체 encoder와 새 분류기**를 BCE로 함께 학습합니다. frozen feature cache는 생성하지 않습니다.

- 원래 fold 1의 **fit 21,032개**로 가중치를 학습합니다.
- **stop 5,288개**의 3-fold threshold 교차평가로 epoch를 고릅니다.
- 같은 stop에서 원래 ver4 **fold 1 단독**도 비교합니다. 이 비교는 5개 모델 앙상블과 구분합니다.
- 별도 Training **calibration 2,880개**의 5-fold 교차평가로 혼합과 임계값을 정합니다.
- 기존 구성보다 Macro F1 **0.002 이상**, **5개 fold 중 3개 이상** 개선될 때 새 구성을 채택합니다.

5개의 새 encoder를 각각 학습하는 OOF 실험은 아닙니다. stop에서 epoch를 반복 비교한 선택 편향도 남습니다.
공개 KoELECTRA의 한국어 표현과 [PLM-ICD의 segment/label attention](https://aclanthology.org/2022.clinicalnlp-1.2.pdf)을
적용한 실험이며, Mission 3에서의 F1 상승이나 총 CU는 실행 결과로 확인해야 합니다.

**중단된 학습 재개:** 같은 코드·데이터·설정, 같은 GPU 종류·AMP 정밀도·관련 패키지 버전을 사용하세요.
**완료된 결과 재사용:** 다른 GPU에서도 가능하며 실행 조건이 다른 예측 캐시는 다시 계산합니다.
새 Colab 세션에서는 입력 ZIP 복원과 공개 encoder 준비를 다시 수행합니다.


In [ ]:
training_started = time.monotonic()
run_python([
    runtime_dir / "train.py", "--train_label_dir", TRAIN_LABEL_DIR,
    "--base_dir", BASE_DIR, "--pretrained_dir", PRETRAINED_DIR,
    "--output_dir", OUTPUT_DIR, "--config_json", config_path, "--auto_batch",
])
TRAINING_MINUTES = (time.monotonic() - training_started) / 60
CHECKPOINT_PATH = OUTPUT_DIR / "mission3_ver7_model.ckpt"
print(f"전체 encoder 학습·내부 비교·보정·저장: {TRAINING_MINUTES:.1f}분")
print("ver7 최종 모델 구성 파일:", CHECKPOINT_PATH)


In [ ]:
# 7. Training 내부 결과. 공식 Validation은 아직 모델 선택에 사용하지 않았습니다.
REPORT_PATH = OUTPUT_DIR / "mission3_ver7_metrics.json"
training_report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
candidate_report = training_report.get("candidate_training", {})
print("KoELECTRA 전체 미세조정 최적 epoch:", candidate_report.get("best_epoch", "상세 기록 확인"))
if candidate_report.get("best_is_initial_baseline", candidate_report.get("best_epoch") == 0):
    print("공개 encoder와 초기 무작위 분류기 상태가 선택됐습니다. 학습한 epoch가 최종 후보에 반영되지 않았습니다.")
print("새 후보 stop 교차평가 Macro F1:", candidate_report.get("best_stop_macro_f1", "상세 기록 확인"))
print("기존 ver4 fold 1 단독 stop Macro F1:",
      training_report.get("baseline_single_fold_stop", {}).get("stop_macro_f1", "상세 기록 확인"))
print("후보 선택 기준:", candidate_report.get("selection_scope", "Training stop threshold 교차평가"))
for field, title in (
    ("candidate_standalone_crossfit_macro_f1", "새 후보 단독 calibration 교차평가"),
    ("baseline_crossfit_macro_f1", "기존 ver4 앙상블 calibration 교차평가"),
    ("candidate_crossfit_macro_f1", "혼합 후보 calibration 교차평가"),
    ("candidate_raw_crossfit_macro_f1", "확률 평균 후보"),
    ("candidate_logit_crossfit_macro_f1", "logit 평균 후보"),
    ("selected_name", "최종 선택"), ("selected_blend_space", "선택된 혼합 방식"),
    ("selection_gain", "calibration F1 개선 폭"), ("minimum_gain", "최소 개선 폭 기준"),
    ("candidate_improved_folds", "개선된 calibration fold 수"),
    ("minimum_improved_folds", "최소 개선 fold 수 기준"),
    ("candidate_fold_gains", "calibration fold별 개선 폭"), ("fallback_reason", "선택 조건 판단"),
):
    if field in training_report:
        print(title + ":", training_report[field])
if training_report.get("selected_name") == "ver4_baseline":
    print("Training 내부 채택 기준을 충족하지 못해 최종 출력에는 기존 구성을 유지합니다.")
selected = training_report.get("calibrator", {})
labels = ["고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란"]
print("증상 | 기존 신경망 alpha | 새 후보 beta | 임계값")
for label, alpha, beta, threshold in zip(labels, selected.get("neural_weights", []),
                                         selected.get("candidate_weights", []), selected.get("thresholds", [])):
    print(f"{label} | {alpha:.2f} | {beta:.2f} | {threshold:.9f}")
print("계산에는 저장된 보정값의 전체 정밀도를 사용합니다. 전체 기록:", REPORT_PATH)


## 8. 확정된 최종 구성과 후보 단독의 Validation 전체 평가

기본 서울 Validation **3,640개 전체**를 사용합니다. 예측 CSV를 확정한 뒤 symptom 정답으로 평가합니다.
`mission3.csv`는 Training에서 선택한 최종 구성, `candidate_only.csv`는 새 encoder 단독 진단입니다.
후보 단독의 임계값도 Training calibration에서 미리 고정합니다. 여기의 두 점수는 모델 선택에 되먹이지 않습니다.
진단을 켜면 최종 출력에서 후보를 사용하지 않는 경우에도 후보의 추가 추론 비용이 필요합니다.
입력·가중치·실행 조건이 같은 원시 예측은 공용 캐시를 재사용합니다.

이전 전체 Validation 기록: **ver4 0.656648948 / ver5 0.656717758 / ver6 0.656669718**.
이전 실행과 현재 GPU·batch 등에 따른 수치 차이가 있을 수 있습니다. 아래 차이는 실행 결과 비교용입니다.


In [ ]:
PREDICTION_PATH = None
CANDIDATE_PREDICTION_PATH = None
evaluation_started = time.monotonic()
run_stamp = datetime.now(timezone(timedelta(hours=9))).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]
if RUN_VALIDATION and VALID_LABEL_DIR is not None:
    prediction_dir = OUTPUT_DIR / "predictions" / run_stamp
    PREDICTION_PATH = prediction_dir / "mission3.csv"
    if RUN_CANDIDATE_DIAGNOSTIC:
        CANDIDATE_PREDICTION_PATH = prediction_dir / "candidate_only.csv"
    run_python([
        runtime_dir / "inference.py", "--audio_dir", "", "--label_dir", VALID_LABEL_DIR,
        "--ckpt_path", CHECKPOINT_PATH, "--output", PREDICTION_PATH,
        *(["--base_dir", BASE_DIR] if not TRAIN_CONFIG.get("bundle_base", False) else []),
        "--batch_size", INFERENCE_BATCH_SIZE, "--cache_dir", INFERENCE_CACHE_DIR, "--evaluate",
        *(["--candidate_output", CANDIDATE_PREDICTION_PATH] if CANDIDATE_PREDICTION_PATH else []),
    ])
    for title, name in [("최종 선택 구성", "validation_metrics.json"),
                        *([("새 후보 단독 (진단용)", "candidate_validation_metrics.json")]
                          if RUN_CANDIDATE_DIAGNOSTIC else [])]:
        metrics_path = prediction_dir / name
        if not metrics_path.is_file():
            print(title + ": CSV 생성 완료. 정답이 없는 입력이면 점수를 생성하지 않습니다.")
            continue
        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
        print(f"{title} — 현재 입력 {metrics['samples']:,}개 Macro F1: {metrics['macro_f1']:.9f}")
        for version, reference in (("ver4", 0.6566489480388843), ("ver5", 0.6567177583831921),
                                   ("ver6", 0.6566697183939796)):
            difference = metrics["macro_f1"] - reference
            print(f"{version} 이전 기록 대비: {difference:+.9f} ({difference * 100:+.5f}%p)")
        print("증상 | Precision | Recall | F1 | 실제 양성 개수")
        for symptom, values in metrics["per_symptom"].items():
            print(f"{symptom} | {values['precision']:.6f} | {values['recall']:.6f} | "
                  f"{values['f1']:.6f} | {values['support']}")
    print("Validation 진단은 저장된 모델·혼합·임계값을 변경하지 않습니다.")
else:
    print("학습·모델 저장 완료. RUN_VALIDATION=False이므로 최종 평가를 생략했습니다.")
EVALUATION_MINUTES = (time.monotonic() - evaluation_started) / 60
timing = {
    "preparation_minutes": PREPARATION_MINUTES,
    "pretrained_preparation_minutes": PRETRAINED_MINUTES,
    "training_and_calibration_minutes": TRAINING_MINUTES,
    "inference_and_evaluation_minutes": EVALUATION_MINUTES,
    "output_dir": str(OUTPUT_DIR),
    "prediction_csv": str(PREDICTION_PATH) if PREDICTION_PATH else None,
    "candidate_prediction_csv": str(CANDIDATE_PREDICTION_PATH) if CANDIDATE_PREDICTION_PATH else None,
    "note": "벽시계 시간이며 CU 측정값이 아닙니다. 패키지 설치 시간은 제외합니다.",
}
timing_path = OUTPUT_DIR / ("notebook_runtime_" + run_stamp + ".json")
timing_path.write_text(json.dumps(timing, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"최종 추론·평가: {EVALUATION_MINUTES:.1f}분")
print("새 ver7 결과:", OUTPUT_DIR)
print("최종 모델 구성:", CHECKPOINT_PATH)
print("최종 예측 CSV:", PREDICTION_PATH)
print("후보 단독 진단 CSV:", CANDIDATE_PREDICTION_PATH)
print("실행 시간 기록:", timing_path)


## 결과 보관

**내 드라이브/DCC_AIdeas/results/mission3/ver7**에 다음 파일이 저장됩니다.

- `mission3_ver7_model.ckpt`: 확정된 모델 구성·혼합·임계값.
- `candidate_best.ckpt`: 학습한 KoELECTRA와 9-label attention 가중치.
- `mission3_ver7_metrics.json`: 후보 학습 이력, 원래 fold 1 비교, 내부 교차평가와 채택 판단.
- `predictions/실행시각/mission3.csv`, `validation_metrics.json`: 최종 예측과 전체·증상별 점수.
- 같은 폴더의 `candidate_only.csv`, `candidate_validation_metrics.json`: 후보 단독 진단.
- `notebook_runtime_실행시각.json`: 준비·모델 다운로드·학습·추론의 실제 시간. CU는 Colab에서 확인합니다.
- `submission/`: 추론 소스, 패키지 목록, README·RESEARCH 문서.

기본 설정은 기존 5개 모델을 다시 복사하지 않습니다. **원래 입력 ZIP과 ver7 결과를 함께 보관하세요.**
Colab 로컬의 복원 파일과 공개 사전학습 파일은 새 세션에서 자동으로 다시 준비합니다.
최종 결과에 포함된 새 후보 가중치로 추론할 때 공개 모델을 다시 다운로드하지 않습니다.

```bash
python submission/inference.py --audio_dir "" --label_dir "JSON 폴더" \
  --ckpt_path mission3_ver7_model.ckpt --base_dir "ZIP에서 복원한 ver4 폴더" --output mission3.csv
```

`--audio_dir`는 대회 인터페이스 호환용입니다. 실행 로그가 남은 노트북도 보관하세요.
